# 🏷️ ITEC Product Dimension — จัดหมวดสินค้าใหม่จากชื่อ

> แทน `ci_item_category.sql` · ออก 5 คอลัมน์เดิม แต่หมวดถูกกว่า
> โปรเจกต์ `scripts/itec/dimension/` · รันได้ทั้ง local · Colab · Kaggle

---

## โจทย์

`CategoryName` / `SubCategoryName` ที่ key มากับ ITEC **ผิดเยอะเกินกว่าจะเชื่อ**
ต้องจัดหมวดใหม่จากสิ่งที่เชื่อได้มากกว่าคือ **ชื่อสินค้า** แล้วออกมาเป็นคอลัมน์เดิม

| เข้า | ออก |
|---|---|
| `ItemName` `CategoryName` `SubCategoryName` `Brand` | `Sale_Type` `Product_Dimension` `Product_Purpose` `Main_Product_Dimension` `Sub_Product_Dimension` |

---

## Flow เต็ม

```
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ① ข้อมูลเข้า                                                        │
 │    dim_item_itec.csv  216,009 แถว                                   │
 │    ItemName · CategoryName · SubCategoryName · Brand · Model/Series  │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ② normalize                                                         │
 │    พิมพ์เล็ก · ตัดอักขระพิเศษ · ยุบช่องว่าง                          │
 │    ⚠️ ต้องเก็บ ก-๛ ไว้ ไม่งั้นภาษาไทยหายหมดโดยไม่มี error            │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ③ กฎ keyword  (Cell 3 ที่เดียว)  →  3 คอลัมน์กลาง                   │
 │                                                                     │
 │    Item_Type      ของชิ้นนี้คืออะไร     Case · Notebook · Charger    │
 │    Item_Host      ใช้กับเครื่องอะไร     Smart Phone · Tablet · PC    │
 │    Item_Platform  แบรนด์/รุ่น           iPhone · Galaxy · Asus       │
 │                                                                     │
 │    ตัดสิน 3 ชั้น — บนสุดเชื่อได้มากสุด                               │
 │      1. ชื่อสินค้า ตรงกับ TYPES                         ~78%        │
 │      2. CategoryName+SubCategory มีคำตรงกับ TYPES        ~11%        │
 │      3. CATEGORY_MAP จับคู่ CategoryName ตรง ๆ           ~11%        │
 │         (ชั้นกันตก · ติดธง by_category=True ไว้ให้ตรวจ)             │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ④ compose()  →  5 คอลัมน์ผลลัพธ์     ← กฎล้วน ไม่มี ML              │
 │                                                                     │
 │    Type เป็นอุปกรณ์เสริม + รู้ Host  →  Main = Host                  │
 │        Case + Smart Phone            →  Sub  = "Smart Phone Case"   │
 │    Type เป็นตัวเครื่อง               →  Main = map ตรง ๆ             │
 │        Notebook                      →  Sub  = "Ordinary-Notebook"  │
 │    ไม่รู้อะไรเลย                     →  "Others"                     │
 │                                                                     │
 │    ✅ Sub คำนวณจาก Main เสมอ → ไม่มีทางขัดกันเอง                    │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ⑤ ตรวจสุขภาพ + เทียบกับ SQL เดิม                                    │
 │    Unknown กี่ % · Others กี่ % · แถวไหนต่างจากเดิม                  │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ⑥ ออกไฟล์ให้คนตรวจ  →  gold set 300 แถว                            │
 │    ⭐ สำคัญที่สุด — ตอนนี้ยังไม่มีใครรู้ว่า "ถูกจริงกี่ %"            │
 └──────────────────────────────┬──────────────────────────────────────┘
                                ▼
 ┌─────────────────────────────────────────────────────────────────────┐
 │ ⑦ ML ยกระดับ  (ทำเมื่อกฎนิ่งแล้ว · ปิดไว้ก่อนด้วยสวิตช์)            │
 │                                                                     │
 │    7.1 supervised   ชื่อ → embedding → LinearSVC → Item_Type        │
 │                     ช่วยแถวที่ชื่อไม่มีคำใบ้ตรง ๆ                    │
 │    7.2 few-shot     ตัวอย่างหมวดละ 20 ชิ้น → centroid → cosine       │
 │                     เพิ่มหมวดใหม่ได้โดยไม่ต้องเทรน                   │
 │    7.3 fine-tune    ปรับ backbone ด้วย contrastive (SetFit)         │
 │                     ทำท้ายสุด ต้อง GPU + label สะอาด                 │
 └─────────────────────────────────────────────────────────────────────┘
```

---

## 🔑 หลักคิด 4 ข้อที่ห้ามลืม

| | |
|---|---|
| **1. ชื่อสินค้าเชื่อได้มากกว่า category** | category เดิมคือสิ่งที่กำลังจะแทน จึงใช้เป็นตัวสำรองเท่านั้น |
| **2. อุปกรณ์เสริมชนะตัวเครื่องเสมอ** | `"เคส iPhone"` คือ **เคส** ไม่ใช่ iPhone → ชั้น 3 ต้องอยู่เหนือชั้น 4 |
| **3. Sub ต้องคำนวณจาก Main** | ถ้าให้ ML เดา Sub ตรง ๆ จะได้ `Main=Tablet` แต่ `Sub=Smart Phone Case` ปนมา |
| **4. ตัวเลขทุกตัวคือ "เหมือนกฎเก่ากี่ %" ไม่ใช่ "ถูกกี่ %"** | จนกว่าจะมี gold set จากข้อ ⑥ |

---

## ⚠️ กับดักที่เคยเจอมาแล้ว — อย่าให้เกิดซ้ำ

| กับดัก | อาการ |
|---|---|
| `[^\w\s]` ไม่มี `ก-๛` | **ภาษาไทยหายทั้งหมด** เงียบ ๆ ไม่มี error · pandas 3 ใช้ RE2 ซึ่ง `\w` เป็น ASCII |
| `LIKE '%pin%'` | จับคำว่า **Pink** — SQL เดิมโดนไป 4,499 แถว |
| `LIKE '%pc%'` | จับ `PCIe` · `HPCB` · `[PC Only]` |
| `"aio "` ใน Cooling | จับ `DESKTOP AIO` และ **`Vaio`** รวม 1,517 แถว |
| `IS_Galaxy` อยู่ในเงื่อนไข Smart Phone | **Galaxy Tab/Watch/Buds 1,045 แถวกลายเป็นมือถือ** |
| `\bmac\b` อยู่ก่อน `mac mini` | Mac mini / Mac Studio กลายเป็น Notebook |
| in-sample evaluation | เอาโมเดลทายข้อมูลที่มันเทรนมา → ตัวเลขสวยหลอก |

---

## ลำดับการรัน

```
รันทีละเซลล์จากบนลงล่าง — Cell 7 ขึ้นไปเป็น ML ปิดสวิตช์ไว้ทั้งหมด

แก้กฎ → รัน Cell 3 → Cell 5 → ดูตัวเลขสุขภาพ → วนซ้ำจนพอใจ → ค่อยไป Cell 6-7
```


## 1 · Setup — ตรวจ environment เอง รันได้ 3 ที่ด้วยไฟล์เดียว

| | ต้องเตรียมอะไร | GPU |
|---|---|---|
| **local** | ไฟล์อยู่ในโฟลเดอร์เดียวกันอยู่แล้ว | ตามเครื่อง |
| **Colab** | อัป 2 ไฟล์ตอนรันเซลล์นี้ | Runtime → Change runtime type → **T4 GPU** |
| **Kaggle** | Add Data 2 ไฟล์เข้า notebook ก่อน | Settings → Accelerator → **GPU T4 x2** |

**2 ไฟล์ที่ต้องเอาไป**

```
itec_dimension_scripts.zip     ← สร้างด้วย  python make_zip.py
_data/dim_item_itec.csv.gz     ← 5.8 MB
```

### Colab
รันเซลล์ล่าง → ขึ้นปุ่มอัปโหลด → **เลือกทั้ง 2 ไฟล์พร้อมกัน** → มันแตก zip กับคลาย gz ให้เอง

### Kaggle
1. `+ Add Input` → `Upload` → ลากทั้ง 2 ไฟล์เข้าไป → Create Dataset
2. `Add` เข้า notebook
3. Run All — เซลล์นี้จะคัดลอกจาก `/kaggle/input` ออกมาที่ `/kaggle/working` ให้เอง

> ⚠️ **ส่วนกฎ (Cell 2-7) ไม่ต้องใช้ GPU เลย** รันบนเครื่องตัวเองได้สบาย
> GPU จำเป็นเฉพาะส่วน ML (Cell 8) เท่านั้น

In [1]:
import os, sys, gzip, shutil, zipfile, subprocess
from pathlib import Path

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()
ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

NEED_CODE = ["dimension.py"]          # rules.yaml สร้างเองได้จาก Cell 3
NEED_DATA = "dim_item_itec.csv"


def _pip(*pkgs):
    """ลงเฉพาะตัวที่ยังไม่มี — Colab/Kaggle มีของพื้นฐานมาให้อยู่แล้ว"""
    miss = []
    for p in pkgs:
        try:
            __import__(p.split("==")[0].replace("-", "_"))
        except ImportError:
            miss.append(p)
    if miss:
        print("ติดตั้ง", " ".join(miss), "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=True)


def _unpack(p: Path):
    """แตกไฟล์ที่เพิ่งได้มา — zip แตก · .gz คลาย · ที่เหลือคัดลอก"""
    p = Path(p)
    if p.suffix == ".zip":
        zipfile.ZipFile(p).extractall(".")
    elif p.name.endswith(".csv.gz"):
        with gzip.open(p, "rb") as f, open(p.name[:-3], "wb") as o:
            shutil.copyfileobj(f, o)
    elif not Path(p.name).exists():
        shutil.copy(p, p.name)


def _missing():
    return ([f for f in NEED_CODE if not Path(f).exists()]
            + ([NEED_DATA] if not (Path(NEED_DATA).exists()
                                   or Path("_data") / NEED_DATA in Path("_data").glob("*")) else []))


# ─────────────────────────────────────────────────────────── local ──
if ENV == "local":
    SCRIPTS = Path.cwd()

# ─────────────────────────────────────────────────────────── colab ──
elif ENV == "colab":
    SCRIPTS = Path("/content"); os.chdir(SCRIPTS)
    _pip("pyyaml", "pandas")
    need = [f for f in NEED_CODE if not Path(f).exists()]
    if not Path(NEED_DATA).exists() and not Path(NEED_DATA + ".gz").exists():
        need.append(NEED_DATA + ".gz")
    if need:
        print("อัปโหลดไฟล์เหล่านี้พร้อมกันได้เลย (เลือกหลายไฟล์ในหน้าต่างเดียว):")
        print("   • itec_dimension_scripts.zip   (dimension.py + rules.yaml)")
        print("   • dim_item_itec.csv.gz          (5.8 MB)")
        from google.colab import files
        for name in files.upload():
            _unpack(Path(name))
    # เผื่ออัป .gz มาแล้วแต่ยังไม่ได้คลาย
    if not Path(NEED_DATA).exists() and Path(NEED_DATA + ".gz").exists():
        _unpack(Path(NEED_DATA + ".gz"))

# ────────────────────────────────────────────────────────── kaggle ──
else:
    SCRIPTS = Path("/kaggle/working"); os.chdir(SCRIPTS)
    _pip("pyyaml")
    # /kaggle/input อ่านอย่างเดียว ต้องคัดลอกออกมาก่อน
    # rglob + is_file() เพราะ dataset ซ้อนโฟลเดอร์หลายชั้น
    # (เคยใช้ glob("/kaggle/input/*/*") แล้วไปโดนโฟลเดอร์ -> IsADirectoryError)
    found = [q for q in Path("/kaggle/input").rglob("*") if q.is_file()]
    print("เจอใน /kaggle/input:", [q.name for q in found][:12])
    for q in found:
        _unpack(q)

# ────────────────────────────────────────────── ตรวจว่าครบไหม ──
missing = [f for f in NEED_CODE if not Path(f).exists()]
DATA = next((p for p in (SCRIPTS / "_data" / NEED_DATA, SCRIPTS / NEED_DATA) if p.exists()), None)
if missing or DATA is None:
    raise FileNotFoundError(
        f"ยังขาด: {missing + ([NEED_DATA] if DATA is None else [])}\n"
        f"  ที่มีอยู่ตอนนี้: {sorted(p.name for p in SCRIPTS.iterdir())[:15]}\n"
        f"  Colab  -> อัป itec_dimension_scripts.zip + dim_item_itec.csv.gz แล้วรันเซลล์นี้ใหม่\n"
        f"  Kaggle -> Add Data ทั้งสองไฟล์เข้า notebook แล้ว Restart & Run All")

sys.path.insert(0, str(SCRIPTS))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
OUT = SCRIPTS / "_out"; OUT.mkdir(exist_ok=True)

import numpy as np, pandas as pd
import dimension as D

try:
    import torch
    GPU = torch.cuda.is_available()
    print("GPU:", torch.cuda.get_device_name(0) if GPU
          else "ไม่มี — ส่วนกฎรันได้ปกติ · ส่วน ML จะช้า (Kaggle แจก T4 ฟรี 30 ชม./สัปดาห์)")
except ImportError:
    GPU = False
    print("ยังไม่มี torch — ไม่เป็นไรถ้ายังไม่ถึงส่วน ML")

print(f"env={ENV} · SCRIPTS={SCRIPTS} · DATA={DATA.name} · pandas {pd.__version__}")


GPU: ไม่มี — ส่วนกฎรันได้ปกติ · ส่วน ML จะช้า (Kaggle แจก T4 ฟรี 30 ชม./สัปดาห์)
env=local · SCRIPTS=C:\Projects\my-first-project\scripts\itec\dimension · DATA=dim_item_itec.csv · pandas 3.0.3


## 2 · โหลดข้อมูล

`normalize_columns()` แปลงชื่อคอลัมน์ให้เป็นชื่อกลาง — CSV ที่ export มาแต่ละครั้งตั้งชื่อไม่เหมือนกัน
(`item_name` / `ItemName` / `Name` → `ItemName`)

In [2]:
df = pd.read_csv(DATA, encoding="utf-8-sig", dtype=str).fillna("")
df = D.normalize_columns(df)

print(f"{len(df):,} แถว · ชื่อไม่ซ้ำ {df.ItemName.nunique():,}")
print("คอลัมน์:", ", ".join(df.columns[:10]))
print()
print("CategoryName 10 อันดับแรก — สังเกตว่ามีของที่ไม่ใช่หมวดสินค้าปนอยู่เยอะ")
print(df.CategoryName.value_counts().head(10).to_string())
df.head(3)

216,009 แถว · ชื่อไม่ซ้ำ 193,512
คอลัมน์: ItemId, base_item_id, ItemName, department, CategoryName, SubCategoryName, ModelSeries, Brand, brand_group, product_status

CategoryName 10 อันดับแรก — สังเกตว่ามีของที่ไม่ใช่หมวดสินค้าปนอยู่เยอะ
CategoryName
Phone Case            22708
PROMO OPERATOR        11977
Notebook              11942
PC Case & Cooling     10780
Smartphone             8609
Phone Accessories      8316
BTB DEMO               7775
PC Core Components     7716
APPLE SERVICE          7280
RESERVE                6696


,ItemId,base_item_id,ItemName,department,CategoryName,SubCategoryName,ModelSeries,Brand,brand_group,product_status,is_valid_item_id
0,** CASE NIPDA 5161 VIVA WW 450 W. SATA,** CASE NIPDA 5161 VIVA WW 450 W. SATA,CASE NIPDA 5161 VIVA WW 45OW.,PC Components,PC Case & Cooling,CASE,,NIPDA,Other,ACTIVE,false
1,.GLOBAL A810,.GLOBAL A810,vv Headphone GLOBAL A810,Audio,Headphone,HEADPHONE,,GLOBAL,Other,ACTIVE,false
2,0 50644 51660 3,0 50644 51660 3,^^ Monster AI 800 MINI-3(Mini jack to Mini jac...,Accessories,Phone Accessories,CABLE,,MONSTER,Other,ACTIVE,false


## 3 · ✏️ กฎ — **แก้ที่นี่ที่เดียว**

รันเซลล์นี้แล้ว `rules.yaml` จะถูกเขียนใหม่ และ `dimension.py` โหลดกฎใหม่ให้อัตโนมัติ

> **ห้ามแก้ `rules.yaml` ด้วยมือ** — มันถูกเขียนทับทุกครั้งที่รันเซลล์นี้

In [ ]:
# ============================================================================
# ✏️ ที่เดียวที่ต้องแก้กฎ — แก้ตรงนี้แล้วรัน cell นี้ rules.yaml จะถูกเขียนใหม่
# ============================================================================
# ลำดับ = ความสำคัญ · ตัวแรกที่ตรงชนะ · ย้ายขึ้นลงเพื่อเปลี่ยนลำดับ
# whole_word True = ต้องเป็นคำเต็ม เช่น stand จะไม่ไปจับ standard
#
# 🔑 TYPES แบ่งเป็น 5 ชั้น ห้ามสลับชั้น (สลับคำในชั้นเดียวกันได้)
#    1 ไม่ใช่ตัวสินค้า   ต้องจับก่อน ไม่งั้นโปรโมชั่นจะกลายเป็นมือถือ
#    2 ชิ้นส่วนคอม
#    3 อุปกรณ์เสริม     ต้องชนะตัวเครื่อง — "เคส iPhone" คือเคส ไม่ใช่ iPhone
#    4 ตัวเครื่อง
#    5 คำอ่อน           คำที่จับผิดง่าย (mac · nb) ต้องอยู่ท้ายสุดเสมอ
#
# 📌 บันทึกจากการไล่อ่าน 1,000 แถวด้วยตา (2026-09-15) — คอมเมนต์ "⚠️ เคยพลาด"
#    คือจุดที่เคยผิดจริงและแก้แล้ว อย่าย้อนกลับ

TYPES = [
    # ── ชั้น 1 · ไม่ใช่ตัวสินค้า ─────────────────────────────────────────
    {"label": "Warranty", "words": ["applecare", "care+", "ประกัน", "insurance", "warranty", "คุ้มครอง"], "whole_word": False},
    # ⚠️ เคยพลาด "BTS ส่วนลด Mac 6,300 + TRUE" -> Notebook · เติม ส่วนลด/จ่ายล่วงหน้า/mnp
    {"label": "Telecom", "words": ["promotion", "โปรโมชั่น", "เปิดเบอร์", "รายเดือน", "เติมเงิน", "trade up", "tradeup",
                                   "walkin", "prebook", "แพ็กเกจ", "ย้ายค่าย", "รายการส่งเสริมการขาย", "แลกซื้อ",
                                   "ส่วนลด", "จ่ายล่วงหน้า", "mnp", "shareplan", "simonly", "เทิร์นเครื่อง",
                                   "unlimited package", "ชำระเงินสด"], "whole_word": False},
    {"label": "Service", "words": ["ค่าบริการ", "ค่าส่ง", "ค่าแรง", "ค่าขนส่ง", "ค่าติดตั้ง", "shipping", "ซ่อม", "repair",
                                   "fee", "service charge", "ค่าธรรมเนียม"], "whole_word": True},
    # แยก part/spare ออกจาก Service — "(Part) LCD" คืออะไหล่ ไม่ใช่ค่าบริการ
    # ⚠️ เคยพลาด "Main FPC Assy" · "Middle Frame Assy" -> Service · เติม assy/fpc
    {"label": "SparePart", "words": ["อะไหล่", "spare", "part", "after sales", "แพรจอ", "ชุดจอ"], "whole_word": True},
    {"label": "SparePart", "words": [" assy", "fpc ", "logic board", "rear system", "back glass"], "whole_word": False},
    # ⚠️ เคยพลาด "EA Game [PC Only]" · "Kaspersky (1 PC)" -> Desktop · Software ต้องมาก่อน
    {"label": "Software", "words": ["software", "license", "microsoft 365", "office 365", "antivirus", "anti-virus",
                                    "internet security", "kaspersky", "norton", "eset ", "mcafee", "windows 11",
                                    "symantec", "endpoint protection", "trend micro", "bitdefender",
                                    "โปรแกรม", "ลิขสิทธิ์"], "whole_word": False},

    # ── ชั้น 2 · ชิ้นส่วนคอม ────────────────────────────────────────────
    {"label": "PowerSupply", "words": ["power supply", "psu ", "เพาเวอร์ซัพพลาย"], "whole_word": False},
    # ไม่มี "aio " — มันไปจับ "DESKTOP AIO" และ "Vaio" รวม 1,517 แถว
    {"label": "Cooling", "words": ["cooling", "cooler", "heatsink", "พัดลม", "ระบายความร้อน", "liquid freezer"], "whole_word": False},
    {"label": "Mainboard", "words": ["mainboard", "motherboard", "เมนบอร์ด", "m/b "], "whole_word": False},
    # ⚠️ เคยพลาด "Gigabyte RX VEGA 56" -> Desktop · เติม rx /vega
    {"label": "GraphicCard", "words": ["graphic card", "vga ", "geforce", "radeon", "rtx ", "gtx ", "rx ", "vega "], "whole_word": False},
    {"label": "RAM", "words": ["ddr4", "ddr5", "so-dimm", "udimm"], "whole_word": False},
    {"label": "RAM", "words": ["ram"], "whole_word": True},
    # ⚠️ เคยพลาด "CPU Intel E4700" -> Desktop · เติม cpu เป็นคำเต็ม
    {"label": "CPU", "words": ["ryzen", "core i3", "core i5", "core i7", "core i9", "ultra 5", "ultra 7", "ultra 9",
                               "athlon", "pentium"], "whole_word": False},
    {"label": "CPU", "words": ["cpu"], "whole_word": True},

    # ⚠️ เคยพลาด "Note book A4 ... [Cover, Kraft]" -> Case · ต้องมาก่อนชั้นอุปกรณ์เสริม
    {"label": "Stationery", "words": ["note book a", "ball pen", "กระดาษ", "ปากกา", "แฟ้ม", "ลวดเย็บ",
                                      "ดินสอ", "สมุด", "stationary"], "whole_word": False},

    # ── ชั้น 3 · อุปกรณ์เสริม ต้องอยู่เหนือตัวเครื่องเสมอ ──────────────────
    # ⚠️ เคยพลาด "Golla Laptop Bags" -> Notebook เพราะ bag ไม่ match "bags"
    {"label": "Bag", "words": ["bag", "bags", "กระเป๋า", "sleeve", "pouch", "backpack", "เป้"], "whole_word": True},
    # ⚠️ Film ต้องมาก่อน Case — "Screen Protector" เคยโดน protec จับเป็น Case
    {"label": "Film", "words": ["film", "ฟิล์ม", "tempered", "กระจกนิรภัย"], "whole_word": True},
    {"label": "Film", "words": ["screen protector", "privacy filter"], "whole_word": False},
    {"label": "Case", "words": ["case", "casing", "เคส", "cover", "ฝาหลัง", "ฝาครอบ", "ซองใส่", "bumper", "กันกระแทก", "skin"], "whole_word": True},
    {"label": "Case", "words": ["protec"], "whole_word": False},
    # Memory ต้องมาก่อน Charger — "MicroSDHC with SD Adapter" ไม่ใช่ที่ชาร์จ
    {"label": "Memory", "words": ["micro sd", "microsd", "sdhc", "sdxc", "memory card", "flash drive", "flashdrive",
                                  "thumb drive", "memory stick"], "whole_word": False},
    {"label": "Adapter", "words": ["card reader", "usb hub", "docking station", "แปลงสัญญาณ", "sound adapter",
                                   "usb adapter", "otg"], "whole_word": False},
    # 🆕 Powerbank แยกจาก Charger — Main ของมันคือ "Powerbank" ไม่ปนกับ Adapter/Charger
    {"label": "Powerbank", "words": ["power bank", "powerbank", "พาวเวอร์แบงค์", "แบตสำรอง"], "whole_word": False},
    {"label": "Charger", "words": ["charger", "adapter", "อะแดปเตอร์", "ที่ชาร์จ", "หัวชาร์จ",
                                   "ปลั๊ก", "plug "], "whole_word": False},
    # ⚠️ เคยพลาด — ถอด "type-c"/"type c" ออก มันจับ "Monitor ... Type-C 90W" กับ "HDD Type-C"
    {"label": "Cable", "words": ["cable", "สายชาร์จ", "สาย usb", "lightning cable", "patch cord"], "whole_word": False},
    {"label": "Strap", "words": ["strap", "สายนาฬิกา", "สายรัด", "band"], "whole_word": True},
    {"label": "Stand", "words": ["stand", "ขาตั้ง", "holder", "ที่วาง", "dock", "แท่นวาง", "tripod"], "whole_word": True},
    {"label": "Keyboard", "words": ["keyboard", "คีย์บอร์ด"], "whole_word": False},
    # ⚠️ เคยพลาด "Corsair Vengence Mice M90" -> Keyboard · เติม mice
    {"label": "Mouse", "words": ["mouse", "mice", "เมาส์", "mousepad"], "whole_word": True},

    # ── ชั้น 4 · ตัวเครื่อง ─────────────────────────────────────────────
    {"label": "Smartwatch", "words": ["smartwatch", "watch series", "watch ultra", "watch se", "galaxy watch",
                                      "watch fit", "นาฬิกาอัจฉริยะ"], "whole_word": False},
    {"label": "Notebook", "words": ["notebook", "netbook", "laptop", "macbook", "vaio", "thinkpad", "ideapad",
                                    "ultrabook", "chromebook", "surface laptop", "surface pro", "โน๊ตบุ๊ค", "โน้ตบุ๊ก"], "whole_word": False},
    {"label": "Desktop", "words": ["desktop", "all-in-one", "computer set", "pc set", "imac", "mac mini", "mac studio",
                                   "mac pro", "matestation", "คอมประกอบ"], "whole_word": False},
    {"label": "Desktop", "words": ["aio"], "whole_word": True},
    # Tablet ต้องมาก่อน Network — "iPad Wi-Fi + Cellular" ไม่ใช่เราเตอร์
    {"label": "Tablet", "words": ["tablet", "ipad", "แท็บเล็ต", "galaxy tab", "matepad", "idea tab"], "whole_word": False},
    {"label": "Tablet", "words": ["tab"], "whole_word": True},
    {"label": "Smartphone", "words": ["smartphone", "สมาร์ทโฟน", "มือถือ", "โทรศัพท์"], "whole_word": False},
    {"label": "Monitor", "words": ["monitor", "จอมอนิเตอร์", "จอคอม"], "whole_word": False},
    {"label": "TV", "words": ["ทีวี", "โทรทัศน์", "bravia", "smart tv"], "whole_word": False},
    {"label": "TV", "words": ["tv"], "whole_word": True},
    {"label": "Printer", "words": ["printer", "เครื่องพิมพ์", "ปริ้นเตอร์", "toner", "หมึกพิมพ์", "ตลับหมึก",
                                   "inkjet", "laserjet", "cartridge"], "whole_word": False},
    {"label": "Projector", "words": ["projector", "โปรเจคเตอร์", "โปรเจกเตอร์"], "whole_word": False},
    # ── ไลฟ์สไตล์ · ต้องมาก่อน Camera/Console ไม่งั้นโดนกลืน ─────────────
    # โดรนกับ GoPro ไปอยู่ Lifestyle & Hobby ไม่ใช่ Camera (ตัดสินใจ 2026-09-16)
    {"label": "DroneCam", "words": ["drone", "โดรน", "mavic", "tello", "gopro", "insta360", "osmo",
                                    "action cam", "action camera"], "whole_word": False},
    {"label": "VR", "words": ["meta quest", "oculus", "gear vr", "vr headset", "vive pro"], "whole_word": False},
    {"label": "PersonalCare", "words": ["hair dryer", "ไดร์เป่าผม", "hair glory", "electric shaver", "โกนหนวด",
                                        "toothbrush", "แปรงสีฟัน", "massage", "นวด", "เครื่องรีดผม", "epilator",
                                        "เครื่องม้วนผม"], "whole_word": False},
    {"label": "SportOutdoor", "words": ["scooter", "สกู๊ตเตอร์", "จักรยาน", "bicycle", "treadmill", "ลู่วิ่ง",
                                        "dumbbell", "ดัมเบล", "skate", "หมวกกันน็อค", "helmet", "อุปกรณ์กีฬา"], "whole_word": False},
    {"label": "HealthDevice", "words": ["smart scale", "เครื่องชั่ง", "blood pressure", "วัดความดัน", "oximeter",
                                        "ปรอทวัด", "smart ring", "เครื่องวัดอุณหภูมิ"], "whole_word": False},

    {"label": "Camera", "words": ["camera", "กล้อง", "webcam", "dslr", "mirrorless", "handycam"], "whole_word": False},
    {"label": "Storage", "words": ["ssd", "hdd", "hard drive", "harddisk", "hard disk", "usb drive", "external drive",
                                   "nas ", "optical drive", "handy drive", "storage", "cd-rw", "dvd-rw"], "whole_word": False},
    # ⚠️ Audio ต้องมาก่อน MusicPlayer — "MP3 XD8 Ear-phone" คือหูฟังของเครื่องเล่น
    #    และ "Ear-phone" มีขีดกลาง ไม่ match earphone จึงต้องใส่ทั้งสองแบบ
    {"label": "Audio", "words": ["headphone", "หูฟัง", "earphone", "ear-phone", "earbud", "speaker", "ลำโพง",
                                 "soundbar", "headset", "airpod", "microphone", "ไมโครโฟน"], "whole_word": False},
    {"label": "Audio", "words": ["buds"], "whole_word": True},
    {"label": "MusicPlayer", "words": ["ipod", "walkman", "mp3 player", "mp4 player"], "whole_word": False},
    {"label": "MusicPlayer", "words": ["mp3", "mp4"], "whole_word": True},
    # ⚠️ เคยพลาด "Gaming Chair" -> Console (มาจาก category Console & Gaming)
    {"label": "Furniture", "words": ["chair", "เก้าอี้", "โต๊ะ", "ชั้นวาง", "โคมไฟ"], "whole_word": False},
    {"label": "Console", "words": ["playstation", "nintendo", "xbox", "console"], "whole_word": False},
    {"label": "Console", "words": ["ps4", "ps5"], "whole_word": True},
    {"label": "Network", "words": ["router", "access point", "wi-fi", "wifi", "mesh", "modem", "rj-11", "rj-45",
                                   "rj11", "rj45", "network", "เราเตอร์", "airport extreme"], "whole_word": False},
    {"label": "HomeAppliance", "words": ["ตู้เย็น", "เครื่องซักผ้า", "เครื่องปรับอากาศ", "แอร์ผนัง", "ไมโครเวฟ", "หม้อทอด",
                                         "เตาอบ", "เตารีด", "เครื่องดูดฝุ่น", "หม้อหุงข้าว", "กาต้มน้ำ", "เครื่องทำน้ำอุ่น",
                                         "เครื่องชงกาแฟ", "appliance"], "whole_word": False},
    {"label": "Battery", "words": ["battery", "แบตเตอรี่", "แบตเตอรี"], "whole_word": False},
    {"label": "UPS", "words": ["ups"], "whole_word": True},

    # ── ชั้น 5 · คำอ่อน ต้องอยู่ท้ายสุดเสมอ ──────────────────────────────
    # ⚠️ เคยพลาดหนัก — "HDD NB WD 500G" -> Notebook · "PQI USB Drive - Mac Silver" -> Notebook
    #    ต้องให้ Storage/Battery/Network/Audio ได้จับก่อน
    {"label": "Notebook", "words": ["nb ", " nb"], "whole_word": False},
    {"label": "Notebook", "words": ["mac"], "whole_word": True},
]

# ════════════════════════════════════════════════════════════════════════
# HOSTS — "ของชิ้นนี้ใช้กับเครื่องตระกูลไหน"
# ════════════════════════════════════════════════════════════════════════
#   Item_Type = Case            บอกว่า "เป็นเคส"
#   Item_Host = Smart Phone     บอกว่า "เคสของมือถือ"   ->  Sub = "Smart Phone Case"
# ลำดับสำคัญ: เจาะจงก่อนกว้าง · Smart Watch/Tablet ต้องมาก่อน Smart Phone
HOSTS = [
    {"label": "Smart Watch", "words": ["apple watch", "applewatch", "watch series", "watch ultra", "galaxy watch",
                                       "smartwatch", "สายนาฬิกา", "for watch", "watch band"], "whole_word": False},
    {"label": "Tablet", "words": ["ipad", "tablet", "แท็บเล็ต", "galaxy tab", "matepad", "idea tab"], "whole_word": False},
    {"label": "Tablet", "words": ["tab"], "whole_word": True},
    {"label": "HeadSet&Earpiece", "words": ["airpod", "earbud", "headphone", "หูฟัง", "earphone", "headset", "buds"], "whole_word": False},
    # 🆕 เจอจาก SubCategoryName เดิมของเคสที่แยกไม่ได้ — CASING-DIGITAL CAMERA · CASING-IPOD TOUCH
    {"label": "Camera", "words": ["digital camera", "กล้อง", "dslr", "mirrorless", "gopro",
                                  "ixus", "powershot", "cybershot"], "whole_word": False},
    {"label": "Music Player", "words": ["ipod", "mp3 player", "walkman"], "whole_word": False},
    {"label": "Notebook", "words": ["macbook", "notebook", "laptop", "thinkpad", "ideapad", "vaio", "surface pro",
                                    "surface laptop", "โน๊ตบุ๊ค", "โน้ตบุ๊ก"], "whole_word": False},
    {"label": "PC", "words": ["desktop", "all-in-one", "computer set", "pc set", "imac", "atx", "tower", "chassis",
                              "เคสคอม"], "whole_word": False},
    {"label": "Smart Phone", "words": ["iphone", "ไอโฟน", "smartphone", "มือถือ", "โทรศัพท์", "galaxy s", "galaxy z",
                                       "galaxy note", "galaxy a", "redmi", "reno", "find x", "vivo y", "oppo a"], "whole_word": False},
]

# แบรนด์บางตัวบอก host ได้แน่นอน — ใช้ตอนที่ชื่อกับ category ไม่บอก
# ไม่ใส่ Galaxy/Xiaomi เพราะเป็นได้ทั้งมือถือ แท็บเล็ต นาฬิกา หูฟัง
HOST_FROM_PLATFORM = {
    "iPhone": "Smart Phone", "iPad": "Tablet", "AppleWatch": "Smart Watch",
    "AirPods": "HeadSet&Earpiece", "MacBook": "Notebook",
}

# ตัวเครื่องเป็น host ของตัวเอง — Notebook ก็คือ host = Notebook
HOST_FROM_TYPE = {
    "Smartphone": "Smart Phone", "Tablet": "Tablet", "Smartwatch": "Smart Watch",
    "Audio": "HeadSet&Earpiece", "Notebook": "Notebook", "Desktop": "PC",
}

# 🆕 เดาชนิดจากแบรนด์ เมื่อชื่อกับ category บอกไม่ได้เลย
# ⚠️ เคยพลาด "Apple iPhone 14 128GB Blue" -> Unknown เพราะ iphone เป็น PLATFORM ไม่ใช่ TYPE
TYPE_FROM_PLATFORM = {
    "iPhone": "Smartphone", "iPad": "Tablet", "AppleWatch": "Smartwatch",
    "AirPods": "Audio", "MacBook": "Notebook",
}

# ════════════════════════════════════════════════════════════════════════
# CATEGORY_FORCE — category ที่เชื่อได้ "มากกว่า" ชื่อสินค้า จึงทับผลจากชื่อ
# ════════════════════════════════════════════════════════════════════════
# ⚠️ ใช้เท่าที่จำเป็นจริง ๆ เท่านั้น — ปกติชื่อสินค้าต้องชนะเสมอ
#    APPLE SERVICE เป็นแคตตาล็อกอะไหล่ทั้งก้อน ชื่อจึงพูดถึงชิ้นส่วน ไม่ใช่สินค้า
#      "Bottom Speaker, iPhone 16"  ชื่อมี Speaker แต่เป็นอะไหล่ ไม่ใช่ลำโพงที่ขาย
#      "Bottom Case, Space Black"   ชื่อมี Case    แต่เป็นฝาล่างโน้ตบุ๊ก
#      "DISPLAY,27IMAC"             ชื่อมี iMac    แต่เป็นจอสำหรับซ่อม
CATEGORY_FORCE = {
    "APPLE SERVICE": ["SparePart"],
}

# ════════════════════════════════════════════════════════════════════════
# CATEGORY_MAP — ชั้นกันตก สำหรับของที่ "ชื่อบอกอะไรไม่ได้เลย"
# ════════════════════════════════════════════════════════════════════════
# ใช้เป็นชั้นสุดท้ายก่อนตก Unknown — ชื่อสินค้ายังชนะเสมอ
#   "CS@ Roborock H7"        ชื่อไม่มีคำใบ้ · CategoryName = Smart Living  -> SmartHome
# ติดธง by_category=True ไว้ทุกแถวที่ผ่านชั้นนี้ เพราะเป็นการยอมเชื่อ category เดิม
#
# ⚠️ ไม่ใส่ category ที่เป็น "สถานะ" ไม่ใช่ "ชนิด" (DEMO/RESERVE)
#    เพราะ Product_Dimension จับ demo ได้อยู่แล้ว และการยัดเป็น NotForSale
#    จะทำให้เสียข้อมูลว่ามันคือสินค้าอะไร ("Apple iPhone 14 128GB" -> NotForSale)
CATEGORY_MAP = {
    # ── อะไหล่ / บริการ / ประกัน ────────────────────────────────
    "SERVICE":               ["Service", ""],
    "APPLE CARE":            ["Warranty", ""],
    "BANANA SURE":           ["Warranty", ""],

    # ── เครือข่าย ──────────────────────────────────────────────
    "PROMO OPERATOR":        ["Telecom", ""],
    "รายการส่งเสริมการขาย":   ["Telecom", ""],

    # ── ตัวเครื่อง ─────────────────────────────────────────────
    "iPhone":                ["Smartphone", "Smart Phone"],
    "IPOD":                  ["MusicPlayer", ""],
    "PROJECTOR":             ["Projector", ""],
    "SECURITY":              ["Camera", ""],
    "SOLAR ROOF TOP":        ["HomeAppliance", ""],
    "Smart Living":          ["SmartHome", ""],
    "HEALTH & SPORT":        ["HealthDevice", ""],
    "OUTDOOR ACTIVITIES":    ["SportOutdoor", ""],
    "TOYS":                  ["Toy", ""],
    "HOBBY":                 ["Toy", ""],
    "STATIONARY":            ["Stationery", ""],

    # ── ของแถม / ของพรีเมียม ───────────────────────────────────
    "GIFT IDEAS":            ["Merchandise", ""],
    "TECHGIFT":              ["Merchandise", ""],
    "PREMIUM":               ["Merchandise", ""],

    # ── อุปกรณ์เสริมทั่วไป (บอก host ได้ด้วย) ──────────────────
    # ⚠️ เคยพลาด — เคย map เป็น Stand ทำให้ "Plug TOSHINO" กลายเป็นขาตั้ง
    #    ใช้ Accessory ซึ่งเป็นหมวดกลาง ๆ ตรงกับ 'Accessory and Others' ของ SQL
    "Phone Accessories":     ["Accessory", "Smart Phone"],
    "Phone Case":            ["Case", "Smart Phone"],
    "APPLE ACC FOR WATCH":   ["Strap", "Smart Watch"],
    "STYLUS":                ["Accessory", "Tablet"],
    "IT ACCESSORIES":        ["Accessory", ""],
    "APP ENABLED-ACCESSORY": ["Accessory", ""],
    "Gaming Gear":           ["Accessory", "PC"],
    "RESERVE":               ["Accessory", ""],
    "PRODUCT BY ORDER":      ["Accessory", ""],
    "ทรัพย์สิน":              ["Accessory", ""],
}

# ════════════════════════════════════════════════════════════════════════
# 🗺️ ตารางประกอบผลลัพธ์ — Item_Type ไหน ไปเป็น Main อะไร
# ════════════════════════════════════════════════════════════════════════
# เดิมฝังอยู่ใน dimension.py · ย้ายมาที่นี่เพื่อให้แก้ได้จากเซลล์เดียว
#
# ── กติกา 2 ข้อ ──────────────────────────────────────────────────────
#  1. อุปกรณ์เสริมที่ "รู้ว่าใช้กับเครื่องอะไร" (host อยู่ใน ACC_HOSTS)
#     -> Main = host          เช่น เคสมือถือ -> Smart Phone
#     -> Sub  = host + คำต่อท้าย จาก ACC_SUFFIX   -> "Smart Phone Case"
#  2. นอกนั้นใช้ MAIN_OF_TYPE ตรง ๆ · ไม่มีในตาราง -> "Others"

# host ที่ให้ต่อท้ายได้ — ตัด Notebook/PC ออกแล้ว
# ⚠️ ของคอม/โน้ตบุ๊กให้ไปรวมที่ IT Accessories แทน ไม่แตกเป็น "Notebook Adapter/Charger"
ACC_HOSTS = ["Smart Phone", "Tablet", "Smart Watch", "HeadSet&Earpiece"]

ACC_SUFFIX = {
    "Case": "Case", "Bag": "Case", "Film": "Film", "Cable": "Cable",
    "Charger": "Adapter/Charger", "Adapter": "Adapter/Charger",
    "Powerbank": "Adapter/Charger",
    "Warranty": "Insurance",
    "Strap": "Other Accessory", "Stand": "Other Accessory",
    "Battery": "Other Accessory", "Memory": "Other Accessory",
}

MAIN_OF_TYPE = {
    # ── ตัวเครื่องหลัก ───────────────────────────────────────────
    "Smartphone": "Smart Phone", "Tablet": "Tablet", "Smartwatch": "Smart Watch",
    "Audio": "HeadSet&Earpiece", "Notebook": "Notebook", "Desktop": "PC",
    "Camera": "Camera", "TV": "TV", "Console": "Console Gaming",
    "Mouse": "Mouse&Keyboard", "Keyboard": "Mouse&Keyboard",
    "Software": "Software", "Warranty": "Insurance", "Telecom": "Telecom Package",

    # ── Appliance ───────────────────────────────────────────────
    # เครื่องใช้ไฟฟ้า + ของใช้ในบ้าน/สุขภาพ + จอ + เครื่องเล่นเพลง
    # รวมจาก: Smart Living · Health & Sport · Music Player · Monitor
    "HomeAppliance": "Appliance",
    "MusicPlayer": "Appliance",     # iPod · Walkman · MP3

    # ── Smart Home ──────────────────────────────────────────────
    # ของที่ติดตั้งในบ้านและคุมผ่านแอป — กลุ่มลูกค้าชัดและโตเร็ว
    "SmartHome": "Smart Home",      # Roborock · Smart Lock · เซ็นเซอร์ · เครื่องฟอกอากาศ

    # ── Personal Care & Health ──────────────────────────────────
    # ของที่ใช้กับร่างกาย — มาร์จิ้นสูง ขายคู่กับ Smart Watch ได้
    "PersonalCare": "Personal Care & Health",   # ไดร์ · แปรงสีฟัน · โกนหนวด · เครื่องนวด
    "HealthDevice": "Personal Care & Health",   # เครื่องชั่ง · วัดความดัน · Smart Ring
    "HealthSport": "Personal Care & Health",

    # ── Lifestyle & Hobby ───────────────────────────────────────
    # ของเล่น พาหนะ งานอดิเรก — ซื้อตามเทศกาล
    "DroneCam": "Lifestyle & Hobby",      # โดรน · GoPro · Insta360
    "SportOutdoor": "Lifestyle & Hobby",  # สกู๊ตเตอร์ · จักรยาน · หมวกกันน็อค
    "VR": "Lifestyle & Hobby",
    "Toy": "Lifestyle & Hobby",

    # ── IT Accessories ──────────────────────────────────────────
    # ชิ้นส่วนคอม + อุปกรณ์เสริมคอม/โน้ตบุ๊ก + อุปกรณ์ต่อพ่วง + อะไหล่
    # รวมจาก: PC&Notebook Component · Projector · Printer · Storage · Network · Spare Part
    # รายละเอียดยังอยู่ครบใน Sub_Product_Dimension (RAM · CPU · Printer · Storage ...)
    "RAM": "IT Accessories", "CPU": "IT Accessories", "Mainboard": "IT Accessories",
    "GraphicCard": "IT Accessories", "Cooling": "IT Accessories",
    "PowerSupply": "IT Accessories", "PC Case": "IT Accessories",
    "Adapter": "IT Accessories", "Cable": "IT Accessories", "Stand": "IT Accessories",
    "Memory": "IT Accessories", "Accessory": "IT Accessories", "UPS": "IT Accessories",
    "Charger": "IT Accessories",          # charger ที่ไม่รู้ว่าของเครื่องอะไร
    "Monitor": "IT Accessories",          # จอคอม — เป็นอุปกรณ์ต่อพ่วง ไม่ใช่ของใช้ในบ้าน
    "Projector": "IT Accessories",
    "Printer": "IT Accessories",
    "Storage": "IT Accessories",
    "Network": "IT Accessories",
    "SparePart": "IT Accessories",

    # ── Powerbank แยกออกมาเดี่ยว ────────────────────────────────
    # ⚠️ เดิมชื่อ "Adapter/Charger/Powerbank" ซึ่งทับกับ "Smart Phone Adapter/Charger"
    "Powerbank": "Powerbank",

    # ── Others · ไม่ใช่สินค้า หรือแยกไม่ได้ ──────────────────────
    "Service": "Service", "Unknown": "Others", "Battery": "Others",
    "Furniture": "Others", "Merchandise": "Others",
    "Stationery": "Others",
    # อุปกรณ์เสริมที่ไม่รู้ว่าของเครื่องอะไร -> แยกไม่ได้จริง ๆ
    "Case": "Others", "Bag": "Others", "Film": "Others", "Strap": "Others",
}

# "ไม่ใช่สินค้าขายจริง" -> คอลัมน์ IS_Product = False
# ⚠️ ใส่ได้ทั้งชื่อ Main และชื่อ Item_Type — เช็คทั้งสองทาง
#    ต้องใส่ชื่อ Item_Type ด้วยเพราะ SparePart ถูกยุบเข้า IT Accessories แล้ว
#    ถ้าเช็คแต่ Main จะกลายเป็นว่าอะไหล่เป็นสินค้าขายจริง
NOT_PRODUCT = ["Others", "Service", "SparePart", "Unknown"]

# ════════════════════════════════════════════════════════════════════════
# 🆕 ยกขึ้นเป็นหมวดหลักของตัวเอง — ทำ "หลัง" คำนวณ Sub เสร็จ
# ════════════════════════════════════════════════════════════════════════
# รายละเอียดเดิมจึงยังอยู่ใน Sub  ("Smart Phone Case" อยู่ใต้ Main "Case")
#
# MAIN_FROM_PLATFORM  ดูจากแบรนด์/รุ่น
#   ⚠️ only_types ขาดไม่ได้ — ถ้าไม่ใส่ กระเป๋า/ฟิล์ม/อะไหล่ของ MacBook
#      จะถูกนับเป็น Mac ไปด้วย (เคยพลาด: Mac Bag 606 · Mac SparePart 141)
MAIN_FROM_PLATFORM = {
    "MacBook": {"main": "Mac", "only_types": ["Notebook", "Desktop"]},
}

# MAIN_FROM_TYPE  ดูจากชนิดสินค้า
#   เดิมเคสมือถือถูกนับเป็น Smart Phone ทำให้ยอดตัวเครื่องเฟ้อเกือบเท่าตัว
#   (Smart Phone 40,696 -> 20,838 หลังแยกเคสออก)
MAIN_FROM_TYPE = {
    "Case": "Case",
}

# เรียงจากเจาะจงไปกว้าง
PLATFORMS = [
    {"label": "AirPods", "words": ["airpod"], "whole_word": False},
    {"label": "AppleWatch", "words": ["apple watch", "applewatch", "watch ultra"], "whole_word": False},
    {"label": "iPhone", "words": ["iphone", "ไอโฟน"], "whole_word": False},
    {"label": "iPad", "words": ["ipad", "ไอแพด"], "whole_word": False},
    {"label": "MacBook", "words": ["macbook", "imac", "mac mini", "mac studio"], "whole_word": False},
    # แยก Samsung ออกจาก Galaxy — printer samsung ไม่ใช่ Galaxy
    {"label": "Galaxy", "words": ["galaxy"], "whole_word": False},
    {"label": "Samsung", "words": ["samsung"], "whole_word": False},
    {"label": "Xiaomi", "words": ["xiaomi", "redmi", "poco"], "whole_word": False},
    {"label": "Huawei", "words": ["huawei"], "whole_word": False},
    # Realme เป็นคนละแบรนด์กับ OPPO (ร่วมกลุ่ม BBK เฉย ๆ)
    {"label": "Realme", "words": ["realme"], "whole_word": False},
    {"label": "OPPO", "words": ["oppo"], "whole_word": False},
    {"label": "Vivo", "words": ["vivo"], "whole_word": True},
    {"label": "Nintendo", "words": ["nintendo"], "whole_word": False},
    {"label": "PlayStation", "words": ["playstation", "ps5", "ps4"], "whole_word": True},
    {"label": "Asus", "words": ["asus", "rog ", "zenbook", "vivobook", "tuf "], "whole_word": False},
    {"label": "Acer", "words": ["acer", "predator", "nitro"], "whole_word": False},
    {"label": "Lenovo", "words": ["lenovo", "thinkpad", "ideapad", "legion"], "whole_word": False},
    {"label": "HP", "words": ["hp ", "pavilion", "omen", "envy", "compaq"], "whole_word": False},
    {"label": "Dell", "words": ["dell", "alienware", "inspiron", "latitude"], "whole_word": False},
    {"label": "MSI", "words": ["msi"], "whole_word": True},
    {"label": "Microsoft", "words": ["microsoft", "surface"], "whole_word": False},
    {"label": "Logitech", "words": ["logitech", "logi "], "whole_word": False},
    {"label": "Razer", "words": ["razer"], "whole_word": False},
    {"label": "JBL", "words": ["jbl"], "whole_word": True},
    {"label": "Anker", "words": ["anker", "soundcore"], "whole_word": False},
    {"label": "Sony", "words": ["sony", "bravia", "cyber-shot", "vaio"], "whole_word": False},
    {"label": "LG", "words": ["lg "], "whole_word": False},
    {"label": "Canon", "words": ["canon"], "whole_word": False},
    {"label": "Garmin", "words": ["garmin"], "whole_word": False},
    {"label": "Brother", "words": ["brother"], "whole_word": False},
    {"label": "Epson", "words": ["epson"], "whole_word": False},
    {"label": "WD-Kingston", "words": ["kingston", "sandisk", "adata", "toshiba", "seagate", "western digital", "wd "], "whole_word": False},
    {"label": "Electrolux", "words": ["electrolux"], "whole_word": False},
]

# ════════════════════════════════════════════════════════════════════════
# DISAMBIGUATE — ได้ประเภทแล้วค่อยดูชื่อสินค้าอีกที
# ════════════════════════════════════════════════════════════════════════
# ⚠️ ดูเฉพาะ "ชื่อสินค้า" ไม่ดู CategoryName
#    เคยดู context ด้วย แล้วเคสมือถือที่ CategoryName = "Bag" กลายเป็น Bag ทั้งกอง
DISAMBIGUATE = [
    # use_context ที่นี่จำเป็น — "CASE NEO 747-BS 450W 24PIN" ชื่อไม่มีคำใบ้เลย
    # ต้องพึ่ง CategoryName = "PC Case & Cooling"
    # ⚠️ เคสคอมรุ่นเก่าไม่มีคำว่า atx เลย — "Case 313-C07-H 300W 2 Fan P4"
    #    ลายเซ็นคือกำลังไฟ + จำนวนพัดลม ต้องดักด้วย ไม่งั้นตกไปกอง Case มือถือ
    {"from": "Case", "words": ["atx", "tower", "chassis", "computer case", "pc case", "เคสคอม", "เคส pc",
                               "case & cooling", "fan p4", "w 2 fan", "w 1 fan", "mid tower", "micro atx",
                               "power 250w", "power 300w", "power 350w", "power 400w", "power 450w",
                               "power 500w", "power 550w", "power 600w", "power 650w", "power 700w"],
     "to": "PC Case", "whole_word": False, "use_context": True},
    # ⚠️ SubCategoryName "CASE CLUB DEMO" คือตู้โชว์ฟิกเกอร์ Hot Toys ไม่ใช่เคสอุปกรณ์
    {"from": "Case", "words": ["case club", "cosbi", "cosbaby"], "to": "Toy",
     "whole_word": False, "use_context": True},
    # ⚠️ คำว่า protec ในกฎ Case ไปดูดฟิล์มกันรอยมาด้วย — "PROGUARD 2.7 LCD Screen"
    {"from": "Case", "words": ["protection lcd", "pre-protection", "lcd screen", "screen protector",
                               "proguard"], "to": "Film", "whole_word": False, "use_context": True},
    {"from": "Case", "words": ["bag", "กระเป๋า", "sleeve", "pouch", "ซองใส่", "carrying"], "to": "Bag", "whole_word": False},
    # ⚠️ "BB-Golla G835 Sunny13 for Macbook" ชื่อมีแต่ Macbook · CategoryName = Bag บอกว่าเป็นกระเป๋า
    {"from": "Notebook", "words": ["bag", "กระเป๋า", "sleeve", "pouch", "backpack"], "to": "Bag",
     "whole_word": True, "use_context": True},
    # ชื่อรุ่นนาฬิกาพูดถึงวัสดุ case / สี band ของตัวเรือน ไม่ใช่ขายเคสหรือสาย
    {"from": "Case", "words": ["watch series", "watch ultra", "watch se", "watch nike+", "smartwatch"], "to": "Smartwatch", "whole_word": False},
    {"from": "Strap", "words": ["watch series", "watch ultra", "smartwatch", "galaxy watch"], "to": "Smartwatch", "whole_word": False},
    # ⚠️ "Cooler Master Computer Case" คือยี่ห้อ ไม่ใช่พัดลม
    {"from": "Cooling", "words": ["computer case", "pc case", "เคสคอม"], "to": "PC Case", "whole_word": False},
    # Charger ที่อยู่ในหมวดเคสคอม คือ Power Supply ไม่ใช่ที่ชาร์จ
    # "Core i5" ในชื่อโน้ตบุ๊กคือสเปก ไม่ใช่ขายซีพียู · เช่นเดียวกับ GeForce ในชื่อ PC
    {"from": "CPU", "words": ["notebook", "laptop", "macbook", "surface", "thinkpad", "ideapad", "vaio"], "to": "Notebook", "whole_word": False},
    {"from": "CPU", "words": ["desktop", "all-in-one", "computer set", "pc set", "imac", "mac mini", "mac studio", "mac pro"], "to": "Desktop", "whole_word": False},
    {"from": "GraphicCard", "words": ["notebook", "laptop", "macbook"], "to": "Notebook", "whole_word": False},
    {"from": "GraphicCard", "words": ["desktop", "all-in-one", "computer set", "pc set"], "to": "Desktop", "whole_word": False},
    # "Magic Keyboard"/"Magic Mouse" ในชื่อ Mac คืออุปกรณ์ติดเครื่อง ไม่ใช่ของที่ขายแยก
    {"from": "Keyboard", "words": ["macbook", "notebook", "laptop"], "to": "Notebook", "whole_word": False},
    {"from": "Mouse", "words": ["macbook", "notebook", "laptop"], "to": "Notebook", "whole_word": False},
    {"from": "Mouse", "words": ["mac pro", "imac", "mac studio", "desktop"], "to": "Desktop", "whole_word": False},
    # ⚠️ "Audio Technica Headphone Professional Monitor Series" ไม่ใช่จอ
    {"from": "Monitor", "words": ["headphone", "earphone", "speaker", "หูฟัง"], "to": "Audio", "whole_word": False},
    # ⚠️ "HP Inkjet Printer All-in-One" ไม่ใช่เดสก์ท็อป
    {"from": "Desktop", "words": ["printer", "inkjet", "laserjet"], "to": "Printer", "whole_word": False},
    # adapter แปลงสัญญาณ ไม่ใช่ที่ชาร์จ
    {"from": "Charger", "words": ["displayport", "hdmi", "dvi", "vga", "ethernet", "converter", "camera adapter",
                                  "sound adapter", "type-c adapter"], "to": "Adapter", "whole_word": False},
]

# ---------------------------------------------------------------------------
# เขียน rules.yaml ให้ dimension.py ใช้ — ห้ามแก้ไฟล์นั้นด้วยมือ
import yaml


def _t(rows):
    return [{"label": r["label"], "words": r["words"],
             "whole_word": r.get("whole_word", False)} for r in rows]


_rules = {
    "types": _t(TYPES),
    "hosts": _t(HOSTS),
    "platforms": _t(PLATFORMS),
    "disambiguate": [{"from": r["from"], "to": r["to"], "words": r["words"],
                      "whole_word": r.get("whole_word", False),
                      "use_context": r.get("use_context", False)} for r in DISAMBIGUATE],
    "host_from_platform": HOST_FROM_PLATFORM,
    "host_from_type": HOST_FROM_TYPE,
    "type_from_platform": TYPE_FROM_PLATFORM,
    "category_map": CATEGORY_MAP,
    "category_force": CATEGORY_FORCE,
    "main_of_type": MAIN_OF_TYPE,
    "acc_suffix": ACC_SUFFIX,
    "acc_hosts": ACC_HOSTS,
    "not_product": NOT_PRODUCT,
    "main_from_platform": MAIN_FROM_PLATFORM,
    "main_from_type": MAIN_FROM_TYPE,
}
with open(SCRIPTS / "rules.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(_rules, f, allow_unicode=True, sort_keys=False, width=120)

nt, nh, npl, nd, ncm = D.reload_rules()
print(f"เขียนกฎแล้ว → {SCRIPTS / 'rules.yaml'}")
print(f"  ประเภท {nt} · host {nh} · แพลตฟอร์ม {npl} · แก้ความกำกวม {nd} · category map {ncm}"
      f" · category force {len(CATEGORY_FORCE)}")
print()
print("ชนิดสินค้า:", " · ".join(dict.fromkeys(r["label"] for r in TYPES)))
print("host      :", " · ".join(dict.fromkeys(r["label"] for r in HOSTS)))
print()
_mains = sorted(set(MAIN_OF_TYPE.values()) | set(ACC_HOSTS)
                | {v["main"] for v in MAIN_FROM_PLATFORM.values()}
                | set(MAIN_FROM_TYPE.values()))
print(f"Main_Product_Dimension ที่เป็นไปได้ {len(_mains)} หมวด:")
for _i in range(0, len(_mains), 4):
    print("   " + "".join(f"{_m:<24}" for _m in _mains[_i:_i+4]))
_miss = sorted({r["label"] for r in TYPES} - set(MAIN_OF_TYPE) - set(ACC_SUFFIX))
if _miss:
    print()
    print(f"⚠️ Item_Type ที่ยังไม่มีใน MAIN_OF_TYPE (จะตกเป็น Others): {_miss}")


## 4 · ทดสอบกฎกับตัวอย่างที่รู้คำตอบ

ทุกบรรทัดคือเคสที่เคยพังมาก่อน — **ถ้าเซลล์นี้ไม่ผ่านแปลว่ากฎถอยหลัง**

In [4]:
CASES = [
    # (ชื่อสินค้า, context, Type ที่ควรได้, Host ที่ควรได้)
    ("iPhone 15 Case Clear",                      "Phone Case",     "Case",       "Smart Phone"),
    ("เคส iPhone 15 ใส กันกระแทก",                 "Phone Case",     "Case",       "Smart Phone"),
    ("Uniq Casing for iPad Mini 4 Gardesuit",     "Phone Case",     "Case",       "Tablet"),
    ("CASE ATX 002",                              "PC Case & Cooling", "PC Case", "PC"),
    ("Sony - Vaio Notebook VPC-EG18FH/B",         "Notebook",       "Notebook",   "Notebook"),
    ("ACER DESKTOP AIO Aspire C24-1650",          "Desktop",        "Desktop",    "PC"),
    ("D8@ Apple Mac mini: M4 chip 10C CPU",       "Mac",            "Desktop",    "PC"),
    ("Apple MacBook Air 13 : M2 chip 8GB",        "Mac",            "Notebook",   "Notebook"),
    ("Garmin Smartwatch Instinct E 40 mm Band",   "Smartwatch",     "Smartwatch", "Smart Watch"),
    ("Apple Watch Ultra 2 GPS 49mm Titanium Case","Smartwatch",     "Smartwatch", "Smart Watch"),
    ("Apple Acc Watch 38mm Pink Sand Sport Band", "APPLE ACC FOR WATCH", "Strap",  "Smart Watch"),
    ("Microsoft Tablet Surface Pro4 Core i5 8GB", "Notebook",       "Notebook",   "Notebook"),
    ("Software EA Game Medal of Honor [PC Only]", "Software",       "Software",   ""),
    ("BTS ส่วนลด Mac 6,300 บาท + TRUE NEW-Umax550","PROMO OPERATOR", "Telecom",   ""),
    ("Lenovo Idea Tab Plus ZAG70716TH Wi-Fi",     "Tablet",         "Tablet",     "Tablet"),
    ("(Part) LCD",                                "APPLE SERVICE",  "SparePart",  ""),
    ("Transportation Fee 40",                     "SERVICE",        "Service",    ""),
    ("printer laser samsung Multifunction SCX4100","Printer",       "Printer",    ""),
    ("ELECTROLUX ตู้เย็น 2 ประตู 14.8Q",           "Home Appliance", "HomeAppliance", ""),
    ("Kingston MicroSDHC 128GB with SD Adapter",  "Storage",        "Memory",     ""),
    ("Apple Acc Mini Displayport to DVI Adapter", "IT ACCESSORIES", "Adapter",    ""),
]

rows, bad = [], 0
for name, ctx, want_t, want_h in CASES:
    t, h, p = D.classify(name, ctx)
    ok = (t == want_t) and (h == want_h)
    bad += (not ok)
    rows.append({"": "✅" if ok else "❌", "ItemName": name[:44],
                 "Type": t, "Host": h or "—", "Platform": p,
                 "ควรได้": "" if ok else f"{want_t} / {want_h or '—'}"})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"ผ่าน {len(CASES)-bad}/{len(CASES)}" + ("  ✅" if bad == 0 else f"  ❌ ตก {bad} เคส"))

                                     ItemName          Type        Host    Platform ควรได้
✅                        iPhone 15 Case Clear          Case Smart Phone      iPhone       
✅                  เคส iPhone 15 ใส กันกระแทก          Case Smart Phone      iPhone       
✅       Uniq Casing for iPad Mini 4 Gardesuit          Case      Tablet        iPad       
✅                                CASE ATX 002       PC Case          PC     Generic       
✅           Sony - Vaio Notebook VPC-EG18FH/B      Notebook    Notebook        Sony       
✅            ACER DESKTOP AIO Aspire C24-1650       Desktop          PC        Acer       
✅         D8@ Apple Mac mini: M4 chip 10C CPU       Desktop          PC     MacBook       
✅          Apple MacBook Air 13 : M2 chip 8GB      Notebook    Notebook     MacBook       
✅     Garmin Smartwatch Instinct E 40 mm Band    Smartwatch Smart Watch      Garmin       
✅  Apple Watch Ultra 2 GPS 49mm Titanium Case    Smartwatch Smart Watch  AppleWatch       

## 5 · รันทั้งชุด → 5 คอลัมน์

`TAXONOMY` เลือกได้ 2 แบบ

| ค่า | ผล |
|---|---|
| `"sql"` | ใช้ค่าชุดเดิมของ SQL เป๊ะ ๆ — report/dashboard เดิมไม่ต้องแก้ |
| `"extended"` | แยก Mac · Case · Service · Powerbank · Smart Home · Lifestyle & Hobby ออกมาเป็นหมวดหลัก |

In [5]:
TAXONOMY = "extended"       # "sql" ถ้า report ปลายทางรับค่าใหม่ไม่ได้

import time
t0 = time.time()
df = D.add_columns(df)
df = D.compose(df, taxonomy=TAXONOMY)
print(f"เสร็จใน {time.time()-t0:.0f}s\n")

print("── สุขภาพของกฎ ──")
print(D.health(df).to_string(index=False))
print()
print("── Main_Product_Dimension ──")
vc = df.Main_Product_Dimension.value_counts()
print(pd.DataFrame({"n": vc, "%": (vc/len(df)*100).round(1)}).to_string())

  ① จับ Item_Type / Host / Platform จากชื่อสินค้า ...
  ② ชื่อบอกไม่ได้ -> ไปดู CategoryName + SubCategoryName ...
  ③ ยังไม่ได้ -> CATEGORY_MAP + เดาจากแบรนด์ ...
  ④ แก้ความกำกวม (DISAMBIGUATE) ...
  ⑤ ตัวเครื่องเป็น host ของตัวเอง ...
เสร็จใน 29s

── สุขภาพของกฎ ──
                            ตัวชี้วัด    แถว     %           เป้า
                  Item_Type = Unknown   3058   1.4          < 10%
                     Item_Host = ว่าง  97057  44.9          < 40%
            Main = Others (แยกไม่ได้)  15186   7.0          < 10%
เชื่อ CategoryName เดิม (by_category)  22022  10.2 ยิ่งน้อยยิ่งดี
              Item_Platform = Generic  99739  46.2              —
                           รวมทั้งหมด 216009 100.0              —

── Main_Product_Dimension ──
                            n     %
Main_Product_Dimension             
Smart Phone             40719  18.9
IT Accessories          32953  15.3
Telecom Package         16931   7.8
Notebook                16481   7.6
Others                 

### ดูรายละเอียดอีกสองมุม

In [6]:
print("── Item_Type (ชนิดสินค้า) ──")
vc = df.Item_Type.value_counts()
print(pd.DataFrame({"n": vc, "%": (vc/len(df)*100).round(1)}).head(25).to_string())
print()
print("── Sub_Product_Dimension 20 อันดับ ──")
print(df.Sub_Product_Dimension.value_counts().head(20).to_string())
print()

# ── ตรวจความสอดคล้อง Main <-> Sub ──────────────────────────────────
# Sub ถูกคำนวณจาก Main เสมอ จึงต้องไม่มี Sub ตัวไหนโผล่ใต้ Main มากกว่า 1 อัน
# (เดิมเช็คว่า Sub ขึ้นต้นด้วย Main ซึ่งใช้ไม่ได้แล้ว เพราะ IT Accessories
#  เก็บรายละเอียดไว้ที่ Sub เป็น RAM / CPU / Mainboard ไม่ได้ขึ้นต้นด้วยชื่อ Main)
dup = df.groupby("Sub_Product_Dimension").Main_Product_Dimension.nunique()
bad = dup[dup > 1]
print(f"Sub ที่โผล่ใต้ Main มากกว่า 1 อัน: {len(bad)}  (ควรเป็น 0)")
if len(bad):
    print(df[df.Sub_Product_Dimension.isin(bad.index)]
            .groupby(["Sub_Product_Dimension", "Main_Product_Dimension"]).size().to_string())
print()
print("── Main -> Sub ที่เป็นไปได้ ──")
for m, g in df.groupby("Main_Product_Dimension"):
    subs = sorted(g.Sub_Product_Dimension.unique())
    head = " · ".join(subs[:6]) + (f"  ...(+{len(subs)-6})" if len(subs) > 6 else "")
    print(f"  {m:<22} {head}")

── Item_Type (ชนิดสินค้า) ──
                   n     %
Item_Type                 
Case           24329  11.3
Telecom        16931   7.8
Notebook       16481   7.6
Audio          13647   6.3
Smartphone     12597   5.8
SparePart      11313   5.2
Desktop         7010   3.2
PC Case         6559   3.0
Accessory       6106   2.8
Storage         6027   2.8
Bag             5566   2.6
Tablet          5299   2.5
Camera          4997   2.3
Film            4784   2.2
HomeAppliance   4735   2.2
Mouse           4696   2.2
Charger         3911   1.8
GraphicCard     3640   1.7
Cable           3564   1.6
Smartwatch      3534   1.6
Monitor         3478   1.6
Unknown         3058   1.4
Printer         2796   1.3
Strap           2786   1.3
Keyboard        2724   1.3

── Sub_Product_Dimension 20 อันดับ ──
Sub_Product_Dimension
Smart Phone Case               19804
Telecom Package                16931
Ordinary-Notebook              14867
HeadSet&Earpiece Main&Other    13647
Smart Phone Main&Other         12

## 6 · แถวที่ยังแยกไม่ได้ — ใช้ตัดสินว่าจะเติมกฎอะไรต่อ

`Item_Type = Unknown` คือกองที่กฎยังไม่มีคำให้จับ
**ดู `CategoryName` ของกองนี้แล้วจะรู้ว่าควรเพิ่มชนิดอะไรเข้า Cell 3**

In [7]:
unk = df[df.Item_Type == "Unknown"]
byc = df[df.by_category]

print(f"① Unknown (ไม่มีอะไรจับได้เลย)        {len(unk):>7,}  {len(unk)/len(df):>6.1%}")
print(f"② เชื่อ CategoryName เดิม (by_category) {len(byc):>7,}  {len(byc)/len(df):>6.1%}")
print()

if len(unk):
    print("── ① CategoryName ของกอง Unknown — เพิ่มเข้า CATEGORY_MAP ได้เลย ──")
    print(unk.CategoryName.value_counts().head(15).to_string())
    print("\nสุ่มมาดู 10 ชื่อ")
    for x in unk.ItemName.sample(min(10, len(unk)), random_state=42):
        print("   ", x[:74])

print("\n── ② กองที่เชื่อ category เดิม — ควรตรวจมากที่สุด ──")
print("(ชื่อสินค้าบอกอะไรไม่ได้ ผลจึงขึ้นกับความถูกของ CategoryName ล้วน ๆ)")
print(byc.groupby(["CategoryName", "Item_Type"]).size().sort_values(ascending=False).head(15).to_string())

① Unknown (ไม่มีอะไรจับได้เลย)          3,058    1.4%
② เชื่อ CategoryName เดิม (by_category)  22,022   10.2%

── ① CategoryName ของกอง Unknown — เพิ่มเข้า CATEGORY_MAP ได้เลย ──
CategoryName
BTB DEMO                    2167
PC Core Components           121
DEMO FS - APPLE              103
Audio & Visual                95
PROJECT SALE                  71
APPLE DEMO                    67
DEMO FS - AUDIO & VISUAL      66
STREAMING                     64
HOME ENTERTAINMENT            52
REALME AIOT                   50
SEASONAL                      46
APPLE                         39
SMILE                         23
GIFT CARD                     22
SELLING EXPENSE               21

สุ่มมาดู 10 ชื่อ
    Xiaomi Amazfit Smart Scale Aurora
    ถุงพลาสติกใส่สินค้า สีส้ม Xiaomi Size L
    EcoFlow Power Hat (EFpowerHat-M-L)
    PP Injection Expanded Luggage W2 Red
    Xiaomi Mi Smart LED Bulb Cool White (26690)
    Apple Acc Watch 38mm Berry Woven Nylon (Demo)
    Ninebot eKickScooter A6 - Green

### เทียบกับ SQL เดิม (ถ้ามีคอลัมน์เดิมติดมาในไฟล์)

ถ้าไฟล์ที่ export มามี `Main_Product_Dimension` เดิมอยู่แล้ว เซลล์นี้จะเทียบให้
ไม่มีก็ข้ามได้ — ไม่ใช่ error

In [8]:
OLD = "old_Main_Product_Dimension"
if OLD in df.columns:
    same = (df[OLD] == df.Main_Product_Dimension).mean()
    print(f"ตรงกับ SQL เดิม {same:.1%}\n")
    d = df[df[OLD] != df.Main_Product_Dimension]
    print(d.groupby([OLD, "Main_Product_Dimension"]).size()
           .sort_values(ascending=False).head(15).to_string())
else:
    print("ไม่มีคอลัมน์ของ SQL เดิมในไฟล์นี้ — ข้ามการเทียบ")
    print("อยากเทียบให้ export SQL เดิมมาเป็นคอลัมน์ชื่อ old_Main_Product_Dimension")

ไม่มีคอลัมน์ของ SQL เดิมในไฟล์นี้ — ข้ามการเทียบ
อยากเทียบให้ export SQL เดิมมาเป็นคอลัมน์ชื่อ old_Main_Product_Dimension


## 7 · ⭐ ออกไฟล์ให้คนตรวจ — gold set

**นี่คือขั้นที่สำคัญที่สุดและถูกข้ามมาตลอด**

ทุกตัวเลขข้างบนบอกแค่ว่า *"กฎทำงานตามที่เขียนไว้"* ไม่ได้บอกว่า *"หมวดที่ได้ถูกต้อง"*
จะรู้ได้ทางเดียวคือให้คนดูแล้วตัดสิน

### ตรวจครบทั้ง 5 คอลัมน์ ไม่ใช่แค่ Main

| คอลัมน์ | ค่าที่เป็นไปได้ | ดูอะไร |
|---|---|---|
| `Sale_Type` | Promotion Sale / Normal Sale | เป็นรายการโปรโมชั่นไหม |
| `Product_Dimension` | Demo Product / Normal Product | เป็นเครื่องโชว์ไหม |
| `Product_Purpose` | Gaming / Ordinary | เป็นของสาย gaming ไหม |
| **`Main_Product_Dimension`** | 23 หมวด | **หมวดหลัก — ตัวสำคัญสุด** |
| `Sub_Product_Dimension` | ~50 หมวด | หมวดย่อย (คำนวณจาก Main) |

### เลือกมา 300 แถว

```
สุ่มคุมสัดส่วน 150   ทุกหมวดมีตัวแทน ไม่งั้นเห็นแต่หมวดใหญ่
น่าสงสัย      150   Unknown · เชื่อ category เดิม · Others · ไม่รู้ host
```

**ทั้งสองกองต้องมี** — กองแรกบอก "ภาพรวมถูกกี่ %" กองสองบอก "จุดที่พังอยู่ตรงไหน"

In [9]:
N_STRAT, N_SUSPECT = 150, 150

# 1) สุ่มแบบคุมสัดส่วน — ทุกหมวดต้องมีตัวแทน ไม่งั้นเห็นแต่หมวดใหญ่
#    ไม่ใช้ groupby.apply เพราะ pandas 3 ตัดคอลัมน์ที่ group ทิ้ง
per = max(1, N_STRAT // max(1, df.Main_Product_Dimension.nunique()) + 1)
parts = [g.sample(min(len(g), per), random_state=42)
         for _, g in df.groupby("Main_Product_Dimension", sort=False)]
strat = pd.concat(parts).sample(min(N_STRAT, sum(map(len, parts))), random_state=42)

# 2) กองที่น่าสงสัย — Unknown / เชื่อ category เดิม / ยังแยกไม่ได้ / ไม่รู้ host
susp_pool = df[(df.Item_Type == "Unknown")
               | df.by_category                      # เชื่อ category เดิม -> เสี่ยงสุด
               | (df.Main_Product_Dimension == "Others")
               | (df.Item_Type.isin(D.COMPOSE["acc_suffix"]) & (df.Item_Host == ""))]
susp = susp_pool.sample(min(N_SUSPECT, len(susp_pool)), random_state=42)

gold = (pd.concat([strat.assign(กลุ่ม="สุ่มทั่วไป"), susp.assign(กลุ่ม="น่าสงสัย")])
          .drop_duplicates("ItemName"))

# ── คอลัมน์ที่ให้คนดู ──────────────────────────────────────────────
SHOW = ["กลุ่ม", "ItemName", "CategoryName", "SubCategoryName", "Brand",
        # คอลัมน์ที่ต้องส่ง — ตรวจให้ครบทุกตัว ไม่ใช่แค่ Main
        "IS_Promotion", "IS_Product",
        "Sale_Type", "Product_Dimension", "Product_Purpose",
        "Main_Product_Dimension", "Sub_Product_Dimension",
        # ชั้นกลาง ไว้ช่วยวินิจฉัยว่าผิดเพราะอะไร
        "Item_Type", "Item_Host", "Item_Platform", "by_category"]
gold = gold[[c for c in SHOW if c in gold.columns]].copy()

# ── ช่องให้กรอก ────────────────────────────────────────────────────
gold["ถูกไหม"] = ""        # y = ถูกครบทั้ง 5 คอลัมน์ · ถ้าผิดข้อใดข้อหนึ่งให้เว้นว่างแล้วกรอกด้านล่าง
gold["Main_ที่ถูก"] = ""   # Main ผิด -> พิมพ์หมวดที่ถูก (ดูรายชื่อที่พิมพ์ออกมาข้างล่าง)
gold["ผิดคอลัมน์อื่น"] = ""  # พิมพ์: sale / demo / gaming / sub  (ใส่หลายตัวคั่นด้วย ,)
gold["หมายเหตุ"] = ""

path = OUT / "gold_set_check.csv"
gold.to_csv(path, index=False, encoding="utf-8-sig")

print(f"เขียน {path} · {len(gold):,} แถว")
print(f"  สุ่มทั่วไป {(gold['กลุ่ม']=='สุ่มทั่วไป').sum():,}"
      f" · น่าสงสัย {(gold['กลุ่ม']=='น่าสงสัย').sum():,}")
print()
print("━━━ วิธีกรอก ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("เปิดด้วย Excel (utf-8-sig อ่านไทยได้เลย) แล้วดู 5 คอลัมน์นี้ทุกแถว")
print("   IS_Promotion       True / False              <- เป็นรายการโปรโมชั่นไหม")
print("   IS_Product         True / False              <- เป็นสินค้าขายจริงไหม")
print("   Sale_Type   Promotion Sale / Normal Sale     <- เหมือน IS_Promotion แต่เป็นข้อความ")
print("   Product_Dimension  Demo Product / Normal     <- เป็นเครื่องโชว์ไหม")
print("   Product_Purpose    Gaming / Ordinary         <- เป็นของสาย gaming ไหม")
print("   Main_Product_Dimension                       <- หมวดหลัก ตัวสำคัญสุด")
print("   Sub_Product_Dimension                        <- หมวดย่อย")
print()
print("ถูกครบทั้ง 5      ->  ถูกไหม = y")
print("Main ผิด          ->  เว้น ถูกไหม ว่าง แล้วพิมพ์หมวดที่ถูกใน Main_ที่ถูก")
print("คอลัมน์อื่นผิด     ->  เว้น ถูกไหม ว่าง แล้วพิมพ์ใน ผิดคอลัมน์อื่น")
print("                      ใช้คำว่า  promo / product / demo / gaming / sub  คั่นด้วย ,")
print()
print("เซฟเป็น  gold_set_checked.csv  แล้วรันเซลล์ถัดไป")
print()
print("━━━ หมวดที่มีให้เลือก (Main_Product_Dimension) ━━━━━━━━━━━━━━━━")
cats = sorted(df.Main_Product_Dimension.unique())
for i in range(0, len(cats), 3):
    print("   " + "".join(f"{c:<28}" for c in cats[i:i+3]))

if ENV == "colab":
    from google.colab import files as _f
    _f.download(str(path))
gold.head(10)


เขียน C:\Projects\my-first-project\scripts\itec\dimension\_out\gold_set_check.csv · 299 แถว
  สุ่มทั่วไป 150 · น่าสงสัย 149

━━━ วิธีกรอก ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
เปิดด้วย Excel (utf-8-sig อ่านไทยได้เลย) แล้วดู 5 คอลัมน์นี้ทุกแถว
   IS_Promotion       True / False              <- เป็นรายการโปรโมชั่นไหม
   IS_Product         True / False              <- เป็นสินค้าขายจริงไหม
   Sale_Type   Promotion Sale / Normal Sale     <- เหมือน IS_Promotion แต่เป็นข้อความ
   Product_Dimension  Demo Product / Normal     <- เป็นเครื่องโชว์ไหม
   Product_Purpose    Gaming / Ordinary         <- เป็นของสาย gaming ไหม
   Main_Product_Dimension                       <- หมวดหลัก ตัวสำคัญสุด
   Sub_Product_Dimension                        <- หมวดย่อย

ถูกครบทั้ง 5      ->  ถูกไหม = y
Main ผิด          ->  เว้น ถูกไหม ว่าง แล้วพิมพ์หมวดที่ถูกใน Main_ที่ถูก
คอลัมน์อื่นผิด     ->  เว้น ถูกไหม ว่าง แล้วพิมพ์ใน ผิดคอลัมน์อื่น
                      ใช้คำว่า  promo / product / demo / gaming

,กลุ่ม,ItemName,CategoryName,SubCategoryName,Brand,IS_Promotion,IS_Product,Sale_Type,Product_Dimension,Product_Purpose,Main_Product_Dimension,Sub_Product_Dimension,Item_Type,Item_Host,Item_Platform,by_category,ถูกไหม,Main_ที่ถูก,ผิดคอลัมน์อื่น,หมายเหตุ
198493,สุ่มทั่วไป,^^ Capdase Soft Jacket2 Xpose for iPod Touch2 ...,Bag,CASING-IPOD TOUCH,CAPDASE,False,True,Normal Sale,Normal Product,Ordinary,Music Player,Music Player,MusicPlayer,,Generic,False,,,,
148015,สุ่มทั่วไป,Apple Macbook Air 11.6/1.4/4/128FLASH - NEW - ...,Mac,MACBOOK AIR,Apple,False,True,Normal Sale,Normal Product,Ordinary,Notebook,Ordinary-Notebook,Notebook,Notebook,MacBook,False,,,,
59869,สุ่มทั่วไป,SAMSUNG QLED TV 49 inch SMART 4K (QA49Q60RAKXXT),Audio & Visual,QLED TV,Samsung,False,True,Normal Sale,Normal Product,Ordinary,TV,TV,TV,,Samsung,False,,,,
190407,สุ่มทั่วไป,WK Power Bank 6000mAh WP008 Sport Car / Drak Blue,Power & Charging,POWER BANK,WK,False,True,Normal Sale,Normal Product,Ordinary,Powerbank,Powerbank,Powerbank,,Generic,False,,,,
102953,สุ่มทั่วไป,Apple Bundle iPad Pro 12.9-inch Wi-Fi 128GB Si...,iPad,APPLE BUNDLE IPAD PRO 12.9-INCH 5TH GEN (2021),Apple,False,True,Normal Sale,Normal Product,Ordinary,Tablet,Tablet Main&Other,Tablet,Tablet,iPad,False,,,,
93699,สุ่มทั่วไป,NW Zyxel HUB 8 port 10/100 Switch ES-108A,Network,CLS,ZYXEL,False,True,Normal Sale,Normal Product,Ordinary,Network,Network,Network,,Generic,False,,,,
31422,สุ่มทั่วไป,PRINTER LASERJER XEROX P215b LED(NEW),Printer,LASER PRINTER,XEROX,False,True,Normal Sale,Normal Product,Ordinary,Printer,Printer,Printer,,Generic,False,,,,
138771,สุ่มทั่วไป,ICARE INSURANCE for Apple MacBook Pro 16,SMILE,INSURANCE,Apple,False,True,Normal Sale,Normal Product,Ordinary,Insurance,Insurance,Warranty,Notebook,MacBook,False,,,,
200363,สุ่มทั่วไป,Printer HP Color LaserJet 2600N,Printer,PRINTER,HP,False,True,Normal Sale,Normal Product,Ordinary,Printer,Printer,Printer,,HP,False,,,,
167142,สุ่มทั่วไป,Philips Hue 10W A60 E27 APR,Smart Living,SMART LIVING PRODUCTS,Philips,False,True,Normal Sale,Normal Product,Ordinary,Smart Living,Smart Living,SmartHome,,Generic,True,,,,


### วัดผลจาก gold set ที่ตรวจแล้ว

รันเซลล์นี้หลังกรอกไฟล์เสร็จ — **นี่คือตัวเลขแรกที่บอกว่า "ถูกจริงกี่ %"**

In [10]:
p = OUT / "gold_set_checked.csv"      # เซฟไฟล์ที่กรอกแล้วเป็นชื่อนี้

if ENV in ("colab", "kaggle") and not p.exists():
    print("อัปไฟล์ที่กรอกแล้วขึ้นมาก่อน (ตั้งชื่อ gold_set_checked.csv)")
    if ENV == "colab":
        from google.colab import files as _f
        import shutil as _s
        for _n in _f.upload():
            _s.copy(_n, p)

if not p.exists():
    print(f"ยังไม่มี {p}")
    print("กรอก gold_set_check.csv แล้วเซฟเป็นชื่อนี้ก่อน")
else:
    g = pd.read_csv(p, encoding="utf-8-sig", dtype=str).fillna("")
    g = g[(g["ถูกไหม"].str.strip() != "")
          | (g["Main_ที่ถูก"].str.strip() != "")
          | (g["ผิดคอลัมน์อื่น"].str.strip() != "")]
    if not len(g):
        raise SystemExit("ไฟล์ยังไม่ได้กรอกอะไรเลย")

    ok_all = g["ถูกไหม"].str.strip().str.lower().eq("y")
    main_bad = g["Main_ที่ถูก"].str.strip() != ""
    other = g["ผิดคอลัมน์อื่น"].str.strip().str.lower()

    print("=" * 62)
    print(f"ตรวจแล้ว {len(g):,} แถว")
    print(f"ถูกครบทั้ง 5 คอลัมน์   {int(ok_all.sum()):>5,}   {ok_all.mean():>6.1%}")
    print("=" * 62)
    print("^ นี่คือตัวเลขแรกที่บอกว่า 'ถูกจริงกี่ %' ไม่ใช่ 'เหมือนกฎกี่ %'")
    print()

    print("── แยกรายคอลัมน์ (ยิ่งสูงยิ่งดี) ──")
    rows = [{"คอลัมน์": "Main_Product_Dimension", "ผิด": int(main_bad.sum()),
             "ถูก %": round((1 - main_bad.mean()) * 100, 1)}]
    for key, col in (("promo", "IS_Promotion"), ("product", "IS_Product"),
                     ("sale", "Sale_Type"), ("demo", "Product_Dimension"),
                     ("gaming", "Product_Purpose"), ("sub", "Sub_Product_Dimension")):
        bad = other.str.contains(key, regex=False)
        rows.append({"คอลัมน์": col, "ผิด": int(bad.sum()),
                     "ถูก %": round((1 - bad.mean()) * 100, 1)})
    print(pd.DataFrame(rows).to_string(index=False))
    print()

    print("── แยกตามกลุ่มที่สุ่มมา ──")
    print("(น่าสงสัยควรต่ำกว่าสุ่มทั่วไป ถ้าไม่ต่ำแปลว่าเลือกกองน่าสงสัยผิด)")
    print(g.assign(ถูก=ok_all).groupby("กลุ่ม").ถูก
           .agg(แถว="size", ถูก="sum", อัตรา="mean").round(3).to_string())
    print()

    if "by_category" in g.columns:
        bc = g.by_category.str.strip().str.lower().eq("true")
        if bc.any():
            print("── แถวที่เชื่อ CategoryName เดิม vs อ่านจากชื่อสินค้า ──")
            print("(ถ้า by_category ถูกน้อยกว่ามาก แปลว่าไม่ควรเชื่อ category เดิม)")
            print(g.assign(ถูก=ok_all, เชื่อcat=bc).groupby("เชื่อcat").ถูก
                   .agg(แถว="size", ถูก="sum", อัตรา="mean").round(3).to_string())
            print()

    if main_bad.any():
        print("── Main ที่ผิด · เอาไปแก้กฎที่ Cell 3 ──")
        print(g[main_bad].groupby(["Main_Product_Dimension", "Main_ที่ถูก"])
               .size().sort_values(ascending=False).head(15).to_string())
        print()
        print("ตัวอย่าง 10 แถวแรกที่ผิด")
        # itertuples ใช้ไม่ได้กับชื่อคอลัมน์ภาษาไทย -> ใช้ iterrows
        for _, r in g[main_bad].head(10).iterrows():
            print(f"   {r['Main_Product_Dimension']:<24} -> {r['Main_ที่ถูก']:<24}"
                  f" | {r['ItemName'][:44]}")
        g[main_bad].to_csv(OUT / "gold_main_wrong.csv", index=False, encoding="utf-8-sig")
        print(f"\nเขียน {OUT / 'gold_main_wrong.csv'} ไว้ไล่แก้กฎ")


ยังไม่มี C:\Projects\my-first-project\scripts\itec\dimension\_out\gold_set_checked.csv
กรอก gold_set_check.csv แล้วเซฟเป็นชื่อนี้ก่อน


---

# 8 · ML ยกระดับ — ปิดสวิตช์ไว้ทั้งหมด

> 🛑 **อย่าเพิ่งรันส่วนนี้จนกว่า `Unknown` จะต่ำกว่า 10% และมี gold set แล้ว**
> ML เรียนจาก label ที่กฎสร้าง — **กฎผิด ML ก็เรียนผิดอย่างมั่นใจ**

## เลือกทางไหน

```
            ┌─ กฎครอบคลุมดีแล้ว (Unknown < 10%) ─┐
            │                                    │
     อยากให้แม่นขึ้น                    อยากเพิ่มหมวดใหม่ง่าย ๆ
            │                                    │
            ▼                                    ▼
      8.1 supervised                       8.2 few-shot
      ชื่อ → embedding → LinearSVC        ตัวอย่าง 20 ชิ้น → centroid
      แม่นสุดในหมวดที่มีข้อมูลเยอะ         เพิ่มหมวดแล้วรันใหม่ 1 นาที
            │                                    │
            └────────────┬───────────────────────┘
                         ▼
                  8.3 fine-tune (SetFit)
                  ปรับ backbone · ต้อง GPU · ทำท้ายสุด
```

## ⚠️ วัดจริงมาแล้ว — อย่าคาดหวังผิด

| วิธี | ตรงกับกฎ | หมายเหตุ |
|---|---|---|
| TF-IDF + centroid | 78.1% | **ไม่ต้องใช้ GPU เลย** |
| embedding + centroid (few-shot) | 79.7% | ต่างจาก TF-IDF แค่ 1.6 จุด |
| zero-shot (คำบรรยายหมวดเปล่า ๆ) | 54.5% | **ใช้ไม่ได้** — ประโยคภาษาคนกับรหัสรุ่นสินค้าอยู่คนละที่ |
| TF-IDF + LinearSVC (เทรนเต็ม) | 96.8% | สูงเพราะ**เลียนแบบกฎ keyword ได้เกือบสมบูรณ์** ไม่ใช่เพราะฉลาด |

> **บนแถวที่กฎจับได้อยู่แล้ว embedding แทบไม่ชนะ TF-IDF**
> ที่ embedding ควรชนะคือแถวที่ชื่อ**ไม่มีคำใบ้** (`Komono Wizard heritage` = นาฬิกา) ซึ่งยังพิสูจน์ไม่ได้จนกว่าจะมี gold set

## 8.0 · โหลด backbone + สร้าง feature (ใช้ร่วมกันทั้ง 8.1 - 8.3)

### feature ไม่ใช่ชื่อสินค้าอย่างเดียวแล้ว

ชื่อสินค้าบางตัวอ่านไม่รู้เรื่องจริง ๆ (`X-9977`, `CS@ 001`) — คอลัมน์อื่นช่วยได้

```
"usb-c cable 1m | หมวด: accessories | ย่อย: cable | แบรนด์: anker"
```

### แต่เทรนแบบเห็นครบทุกแถวไม่ได้

ของใหม่ที่เพิ่งเข้าระบบ **ยังไม่มีใคร key หมวดให้** เหลือแค่ `ItemName`
ถ้าโมเดลเคยเห็นแต่ข้อมูลครบ พอเจอของจริงจะพัง — เรียกว่า **train-serving skew**

วิธีแก้คือ **สุ่มปิด field ตอนเทรน** โมเดลตัวเดียวชินทั้งสองสภาพ

| field | โอกาสถูกตัดทิ้ง | เหตุผล |
|---|---|---|
| `ItemName` | **0%** | สิ่งเดียวที่มีเสมอ |
| `CategoryName` | 50% | เชื่อไม่ได้เต็มร้อย และของใหม่มักไม่มี |
| `SubCategoryName` | 50% | เหมือนกัน |
| `Brand` | 30% | หายบ่อยน้อยกว่า |

### 🛑 กับดักที่ต้องกันไว้ — target leakage

`by_category = True` คือแถวที่ **กฎตัดสินจาก `CategoryName`** (~10% ของข้อมูล)
label ของแถวพวกนั้นจึงเป็นฟังก์ชันตรง ๆ ของคอลัมน์นั้น

ถ้าปล่อยให้โมเดลเห็น `CategoryName` ด้วย มันจะแค่ **อ่านค่าเดิมคืนมา**
accuracy พุ่งสวยแต่ไม่ได้เรียนอะไรเลย

→ `build_text()` **บังคับตัด `CategoryName` ทิ้ง 100% เฉพาะแถวเหล่านั้น** ไม่ว่าจะ drop หรือไม่

> โน้ตเก่า `ITEC Category Model (ML)` ในvault ติดกับดักนี้ไว้เต็ม ๆ อย่าลอกไปใช้


In [ ]:
RUN_ML = False        # ⬅️ เปิดเมื่อพร้อม · bge-m3 ดาวน์โหลด ~2.3 GB

if not RUN_ML:
    print("ข้ามส่วน ML · ตั้ง RUN_ML = True แล้วรันใหม่เมื่อพร้อม")
else:
    # Colab/Kaggle ไม่มี sentence-transformers มาให้ ต้องลงเอง (~2 นาที)
    _pip("sentence-transformers")
    from sentence_transformers import SentenceTransformer
    from sklearn.model_selection import train_test_split

    # bge-m3 ดีกว่าแต่ใหญ่ · CPU ให้ใช้ MiniLM ไม่งั้นรอเป็นชั่วโมง
    BACKBONE = "BAAI/bge-m3" if GPU else "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    embedder = SentenceTransformer(BACKBONE, device="cuda" if GPU else "cpu")
    embedder.max_seq_length = 96       # ต่อ 4 field แล้วยาวขึ้น · เดิมชื่อล้วนใช้ 64 พอ
    print(f"backbone = {BACKBONE} · device = {'cuda' if GPU else 'cpu'}")

    LABEL_COL = "Item_Type"            # เริ่มที่ชนิดล้วนเสมอ หมวดน้อย ตัวอย่างต่อหมวดเยอะ
    work = df[df[LABEL_COL] != "Unknown"].reset_index(drop=True)
    vc = work[LABEL_COL].value_counts()
    work = work[work[LABEL_COL].isin(vc[vc >= 20].index)].reset_index(drop=True)
    Y = work[LABEL_COL].to_numpy(dtype=object)

    # ⚠️ แบ่ง train/test ก่อนสร้างข้อความ
    #    เพราะชุดทดสอบต้องสร้างได้ 2 สภาพจากแถวเดียวกัน
    TR, TE = train_test_split(np.arange(len(work)), test_size=0.2,
                              random_state=42, stratify=Y)

    TXT_TR   = D.build_text(work.iloc[TR], drop=True)            # สุ่มปิด field -> ใช้เทรน
    TXT_FULL = D.build_text(work.iloc[TE], drop=False)           # ข้อมูลครบ
    TXT_NAME = D.normalize(work.iloc[TE].ItemName).tolist()      # ชื่อล้วน

    assert len(TXT_TR) == len(TR) and len(TXT_FULL) == len(TE) == len(TXT_NAME)

    n_mask = int(work.iloc[TR].by_category.sum()) if "by_category" in work.columns else 0
    print(f"\nข้อมูลสำหรับ ML: {len(work):,} แถว · {pd.Series(Y).nunique()} หมวด")
    print(f"  train {len(TR):,} · test {len(TE):,}")
    print(f"  แถวที่บังคับซ่อน CategoryName (by_category) ในชุดเทรน: {n_mask:,}")

    print("\n── ตัวอย่างข้อความที่ป้อนเข้าโมเดล ──")
    for t in TXT_TR[:5]:
        print("   ", t[:110])

    if not GPU:
        print(f"\n⚠️ ไม่มี GPU — encode {len(work):,} แถวบน CPU ใช้เวลาประมาณ "
              f"{len(work)/1000*1.2:.0f} นาที  (ลดขนาดก่อนได้ด้วย work = work.sample(20000))")


## 8.1 · Supervised — embedding → LinearSVC

ตรงไปตรงมาที่สุด · แม่นสุดในหมวดที่มีข้อมูลเยอะ

**วัดผล 2 สภาพจากชุดทดสอบเดียวกัน** — ตัวเลขเดียวบอกไม่พอ

| สภาพ | ตรงกับงานจริงตอนไหน |
|---|---|
| ข้อมูลครบ | จัดหมวดของเก่า 216k แถวที่มี category ติดมาแล้ว |
| ชื่อล้วน | ของใหม่เข้าระบบพรุ่งนี้ ยังไม่มีใคร key หมวด |

**ช่องว่างระหว่าง 2 บรรทัด = ราคาที่จ่ายเมื่อไม่มี category ให้ดู**
ถ้าห่างกันมาก แปลว่าโมเดลพึ่ง category มากไป → เพิ่มอัตรา drop ใน `D.ML_FIELDS`


In [ ]:
RUN_SUPERVISED = False

if not (RUN_ML and RUN_SUPERVISED):
    print("ข้าม 8.1 · ต้องตั้ง RUN_ML และ RUN_SUPERVISED เป็น True")
else:
    from sklearn.svm import LinearSVC
    from sklearn.metrics import accuracy_score, f1_score

    def enc(t):
        return embedder.encode(t, batch_size=64, normalize_embeddings=True,
                               show_progress_bar=True)

    clf = LinearSVC(C=1.0, class_weight="balanced").fit(enc(TXT_TR), Y[TR])

    rows = []
    for label, txt in (("ข้อมูลครบ (จัดหมวดของเก่า)", TXT_FULL),
                       ("ชื่อล้วน (ของใหม่ยังไม่ key)", TXT_NAME)):
        p = clf.predict(enc(txt))
        rows.append({"สภาพตอนใช้งาน": label,
                     "accuracy": round(accuracy_score(Y[TE], p), 4),
                     "f1_macro": round(f1_score(Y[TE], p, average="macro"), 4)})
    r = pd.DataFrame(rows)
    gap = r.f1_macro.iloc[0] - r.f1_macro.iloc[1]

    print("\n" + r.to_string(index=False))
    print(f"\nช่องว่าง f1_macro = {gap:.4f}   <- ราคาที่จ่ายเมื่อไม่มี category ให้ดู")
    print("^ ตัวเลขพวกนี้อ่านว่า 'เลียนแบบกฎได้กี่ %' ไม่ใช่ 'ถูกกี่ %'")
    print("  จะรู้ว่าถูกจริงไหม ต้องวัดกับ gold set ที่คนตรวจแล้ว (ส่วน 7)")

    import joblib
    joblib.dump({"backbone": BACKBONE, "label_col": LABEL_COL, "clf": clf,
                 "classes": sorted(set(Y)), "ml_fields": D.ML_FIELDS,
                 "max_seq_length": embedder.max_seq_length},
                OUT / "model_item_type.joblib")
    print(f"\nเซฟ {OUT / 'model_item_type.joblib'}")
    print("ตอนเรียกใช้ต้องสร้างข้อความด้วย D.build_text(df, drop=False) เท่านั้น")


## 8.2 · Few-shot prototype — ไม่ต้องเทรน

`prototype` = ค่าเฉลี่ยเวกเตอร์ของตัวอย่างจริงในหมวดนั้น · ตัดสินด้วยระยะทาง

**จุดขาย: เพิ่มหมวดใหม่ = ใส่ตัวอย่าง 20 ชื่อแล้วรันใหม่** ไม่ต้องเทรน ไม่ต้องแก้ keyword

⚠️ **ต้องใช้ชื่อสินค้าจริงเป็นตัวอย่าง ห้ามใช้คำบรรยายหมวด** — วัดแล้วต่างกัน 25 จุด (79.7% vs 54.5%)

`margin` (ห่างจากอันดับสอง) เป็นตัวคัดที่ดีกว่า `score` — ที่ความครอบคลุมพอกัน แม่นกว่า 6 จุด

In [ ]:
RUN_FEWSHOT = False
K_PROTO = 20

if not (RUN_ML and RUN_FEWSHOT):
    print("ข้าม 8.2 · ต้องตั้ง RUN_ML และ RUN_FEWSHOT เป็น True")
else:
    rng = np.random.default_rng(42)
    cats, proto_vecs = [], []
    for t, g in work.groupby(LABEL_COL):
        if len(g) < K_PROTO:
            continue
        ex = D.build_text(g.sample(K_PROTO, random_state=42), drop=False)
        v = embedder.encode(ex, normalize_embeddings=True).mean(0)
        cats.append(t); proto_vecs.append(v)
    P = np.stack(proto_vecs)
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    print(f"สร้าง prototype {len(cats)} หมวด · หมวดละ {K_PROTO} ชื่อ")

    def fewshot(sub, top=2):
        """sub = DataFrame ไม่ใช่ list ชื่อ — ต้องมีคอลัมน์อื่นให้ build_text ด้วย"""
        names = sub.ItemName.tolist()
        v = embedder.encode(D.build_text(sub, drop=False),
                            batch_size=64, normalize_embeddings=True,
                            show_progress_bar=len(names) > 500)
        sim = np.asarray(v) @ P.T
        order = np.argsort(-sim, axis=1)
        i = np.arange(len(sim))
        return pd.DataFrame({
            "ItemName": list(names),
            "pred":   [cats[j] for j in order[:, 0]],
            "score":  sim[i, order[:, 0]].round(4),
            "margin": (sim[i, order[:, 0]] - sim[i, order[:, 1]]).round(4),
            "pred_2": [cats[j] for j in order[:, 1]],
        })

    # ยิงกับกองที่กฎแยกไม่ได้ — จุดที่ few-shot น่าจะช่วยที่สุด
    unk = df[df.Item_Type == "Unknown"]
    sample = unk.sample(min(2000, len(unk)), random_state=42)
    res = fewshot(sample)
    res.sort_values("margin", ascending=False).to_csv(OUT / "fewshot_unknown.csv",
                                                      index=False, encoding="utf-8-sig")
    print(f"\nเขียน {OUT / 'fewshot_unknown.csv'}")
    for th in (0.02, 0.05, 0.10):
        k = res.margin >= th
        print(f"  margin >= {th:.2f}  เชื่อได้ {k.mean():.1%} ของกอง")
    print("\nที่มั่นใจสุด 15 แถว — เอาไปเขียนเป็นกฎใหม่ใน Cell 3 ได้เลย")
    print(res.nlargest(15, "margin")[["pred", "margin", "ItemName"]].to_string(index=False))

## 8.3 · Fine-tune backbone (SetFit)

ปรับ backbone ให้สินค้าหมวดเดียวกันมีเวกเตอร์ใกล้กัน — **เปลี่ยนแค่ชั้นแรก classifier ยังตัวเดิม**
ต่างกันชั้นเดียวจึงชี้สาเหตุได้ว่าที่ดีขึ้น/แย่ลงมาจากอะไร

### 🛑 เช็กก่อน ไม่ครบอย่าเพิ่งรัน

| # | เงื่อนไข |
|---|---|
| 1 | **มี GPU** — CPU ช้าจนไม่คุ้ม ไปใช้ Kaggle T4 |
| 2 | **`Unknown` < 10%** — ไม่งั้นสอนโมเดลด้วย label ที่ยังไม่นิ่ง |
| 3 | **มี gold set แล้ว** — ไม่งั้นวัดไม่ได้ว่าดีขึ้นจริงหรือแค่เหมือนกฎมากขึ้น |

### ขั้นตอน

1. ตั้ง `RUN_FT = True` แล้วรัน
2. รอ ~10-15 นาที (MiniLM / 20k แถว บน T4) · bge ~40 นาที และอาจ OOM
3. อ่าน `f1_macro ต่าง` **ไม่ใช่ accuracy** เพราะหมวดเบ้มาก
   - `> +0.01` ✅ คุ้ม → เอา backbone ใหม่ไปใส่ Cell 8.0
   - `±0.01` ⚠️ ไม่คุ้ม → กลับไปทำ label ให้สะอาด
   - `< -0.01` 🛑 แย่ลง → **label ยัง noisy** กลับไปแก้กฎ Cell 3
4. อยากดันต่อค่อยขยับทีละอย่าง: `N` 20k→50k → MiniLM→bge → epoch 1→2

🔑 **`batch_sampler=GROUP_BY_LABEL` ขาดไม่ได้** — ถ้า batch มีหมวดละ 1 ตัว
จะสร้างคู่บวกไม่ได้เลย → `loss = 0` → **รันผ่านแต่ไม่เรียนอะไร**

In [ ]:
RUN_FT   = False
FT_N     = 20000      # 20k พอบอกทิศทาง อย่ารันเต็มตั้งแต่รอบแรก
FT_EPOCH = 1          # เกิน 1 เสี่ยงจำข้อมูลแทนเรียน pattern
FT_BATCH = 32         # ต้องใหญ่พอให้มีหลายหมวดต่อ batch
FT_LR    = 2e-5       # มาตรฐาน transformer · สูงกว่านี้ทำลายของเดิมที่โมเดลรู้
FT_MINPC = 4          # หมวดที่มีน้อยกว่านี้ สร้างคู่บวกไม่ได้

if not (RUN_ML and RUN_FT):
    print("ข้าม 8.3 · ต้องตั้ง RUN_ML และ RUN_FT เป็น True")
else:
    assert GPU, "fine-tune บน CPU ช้าจนไม่คุ้ม — ไปรันบน Kaggle T4"
    for _p in ("datasets", "accelerate"):
        try:
            __import__(_p)
        except ImportError:
            print(f"ติดตั้ง {_p} ..."); subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=True)

    import time
    from datasets import Dataset
    from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                       SentenceTransformerTrainingArguments)
    from sentence_transformers.losses import BatchAllTripletLoss
    from sentence_transformers.training_args import BatchSamplers
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score

    sub = work.sample(min(FT_N, len(work)), random_state=42).reset_index(drop=True)
    vc = sub[LABEL_COL].value_counts()
    sub = sub[sub[LABEL_COL].isin(vc[vc >= FT_MINPC].index)].reset_index(drop=True)
    txt = D.build_text(sub, drop=True)        # feature เดียวกับ 8.1
    y = sub[LABEL_COL].to_numpy(dtype=object)
    tr, te = train_test_split(np.arange(len(sub)), test_size=0.2, random_state=42, stratify=y)
    print(f"{len(sub):,} แถว · {pd.Series(y).nunique()} หมวด · เฉลี่ยหมวดละ "
          f"{len(sub)/pd.Series(y).nunique():.0f}   <- ยิ่งน้อย triplet ยิ่งหาคู่บวกไม่เจอ")

    def ft_score(m, tag):
        V = m.encode(txt, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        c = LinearSVC(C=1.0, class_weight="balanced").fit(V[tr], y[tr])
        p = c.predict(V[te])
        a, f = accuracy_score(y[te], p), f1_score(y[te], p, average="macro")
        print(f"  [{tag}] acc={a:.4f} f1_macro={f:.4f}")
        return a, f

    print("\n① baseline · backbone แช่แข็ง")
    base = SentenceTransformer(BACKBONE, device="cuda")
    base.max_seq_length = 64          # ต้องเท่ากับตัว fine-tune ไม่งั้นเทียบไม่ยุติธรรม
    acc0, f10 = ft_score(base, "frozen")

    # fine-tune เห็นแต่ train เท่านั้น — ให้เห็น test ด้วยคะแนนจะสวยแบบหลอกตัวเอง
    print(f"\n② fine-tune · {FT_EPOCH} epoch · batch {FT_BATCH} · lr {FT_LR}")
    codes, _ = pd.factorize(pd.Series(y[tr]))        # triplet loss ต้องการ label เป็น int
    ds = Dataset.from_dict({"sentence": [txt[i] for i in tr], "label": codes.tolist()})

    ft = SentenceTransformer(BACKBONE, device="cuda")
    ft.max_seq_length = 64
    targs = SentenceTransformerTrainingArguments(
        output_dir=str(OUT / "ft-ckpt"), num_train_epochs=FT_EPOCH,
        per_device_train_batch_size=FT_BATCH, learning_rate=FT_LR,
        warmup_ratio=0.1, fp16=True,
        batch_sampler=BatchSamplers.GROUP_BY_LABEL,   # ⭐ ขาดไม่ได้
        logging_steps=100, save_strategy="no", report_to=[])
    t0 = time.time()
    SentenceTransformerTrainer(model=ft, args=targs, train_dataset=ds,
                               loss=BatchAllTripletLoss(ft)).train()
    print(f"  ใช้เวลา {time.time()-t0:.0f}s")
    ft_dir = OUT / f"{LABEL_COL}-finetuned"
    ft.save(str(ft_dir)); print(f"  เซฟ -> {ft_dir}")

    print("\n③ หลัง fine-tune · test ชุดเดิม seed เดิม classifier เดิม")
    acc1, f11 = ft_score(ft, "fine-tuned")

    d_acc, d_f1 = acc1 - acc0, f11 - f10
    print("\n" + "=" * 56)
    print(f"{'':18}{'accuracy':>12}{'f1_macro':>12}")
    print(f"{'frozen':18}{acc0:>12.4f}{f10:>12.4f}")
    print(f"{'fine-tuned':18}{acc1:>12.4f}{f11:>12.4f}")
    print(f"{'ต่าง':18}{d_acc:>+12.4f}{d_f1:>+12.4f}")
    print("=" * 56)
    print("✅ คุ้ม · เอา backbone ใหม่ไปใส่ Cell 8.0" if d_f1 > 0.01 else
          "⚠️ ไม่คุ้ม · กลับไปทำ label ให้สะอาด (Cell 3)" if d_f1 > -0.01 else
          "🛑 แย่ลง · label ยัง noisy เกินไป ต้องแก้กฎก่อน")
    pd.DataFrame([{"backbone": BACKBONE, "label": LABEL_COL, "n": len(sub),
                   "epochs": FT_EPOCH, "acc_frozen": acc0, "f1_frozen": f10,
                   "acc_ft": acc1, "f1_ft": f11, "d_f1": d_f1}]
                 ).to_csv(OUT / "finetune_result.csv", index=False, encoding="utf-8-sig")

---

## 9 · ส่งออก

เก็บทั้ง 3 คอลัมน์กลาง (`Item_Type` `Item_Host` `Item_Platform`) ไว้ด้วย
— ไว้ทำ filter / search และไว้สืบย้อนว่าทำไมแถวนี้ถึงได้หมวดนี้

In [15]:
KEEP = ["ItemId", "ItemName", "CategoryName", "SubCategoryName", "Brand",
        "Item_Type", "Item_Host", "Item_Platform", "by_category",
        "Sale_Type", "Product_Dimension", "Product_Purpose",
        "Main_Product_Dimension", "Sub_Product_Dimension"]
out = df[[c for c in KEEP if c in df.columns]]

path = OUT / "itec_dimension.csv"
out.to_csv(path, index=False, encoding="utf-8-sig")
print(f"เขียน {path} · {len(out):,} แถว · {len(out.columns)} คอลัมน์")

try:
    out.to_parquet(OUT / "itec_dimension.parquet", index=False)
    print(f"เขียน {OUT / 'itec_dimension.parquet'}  (เล็กกว่ามาก เอาขึ้น S3/Glue ได้เลย)")
except Exception as e:
    print("ข้าม parquet:", e)

if ENV in ("colab", "kaggle"):
    import zipfile as _z
    zp = OUT / "itec_dimension_result.zip"
    with _z.ZipFile(zp, "w", _z.ZIP_DEFLATED) as z:
        for f in sorted(OUT.glob("*.csv")) + sorted(OUT.glob("*.parquet")):
            z.write(f, f.name)
    print(f"\nรวมเป็น {zp} ({zp.stat().st_size/1e6:.1f} MB)")
    if ENV == "colab":
        from google.colab import files as _f
        _f.download(str(zp))
    else:
        print("Kaggle: ดาวน์โหลดจากแถบ Output ทางขวา (ต้อง Save Version ก่อน)")
out.head(10)

เขียน C:\Projects\my-first-project\scripts\itec\dimension\_out\itec_dimension.csv · 216,009 แถว · 14 คอลัมน์
เขียน C:\Projects\my-first-project\scripts\itec\dimension\_out\itec_dimension.parquet  (เล็กกว่ามาก เอาขึ้น S3/Glue ได้เลย)


,ItemId,ItemName,CategoryName,SubCategoryName,Brand,Item_Type,Item_Host,Item_Platform,by_category,Sale_Type,Product_Dimension,Product_Purpose,Main_Product_Dimension,Sub_Product_Dimension
0,** CASE NIPDA 5161 VIVA WW 450 W. SATA,CASE NIPDA 5161 VIVA WW 45OW.,PC Case & Cooling,CASE,NIPDA,PC Case,,Generic,False,Normal Sale,Normal Product,Ordinary,IT Accessories,PC Case
1,.GLOBAL A810,vv Headphone GLOBAL A810,Headphone,HEADPHONE,GLOBAL,Audio,HeadSet&Earpiece,Generic,False,Normal Sale,Normal Product,Ordinary,HeadSet&Earpiece,HeadSet&Earpiece Main&Other
2,0 50644 51660 3,^^ Monster AI 800 MINI-3(Mini jack to Mini jac...,Phone Accessories,CABLE,MONSTER,Cable,Smart Phone,Generic,False,Normal Sale,Normal Product,Ordinary,Smart Phone,Smart Phone Cable
3,0 50644 51663 4,^^ Monster AI 1000 Y - SPLT(Y-Splitter),Phone Accessories,CABLE,MONSTER,Cable,Smart Phone,Generic,False,Normal Sale,Normal Product,Ordinary,Smart Phone,Smart Phone Cable
4,0%-10M-CASH-BACK-6780,โปรโมชั่นผ่อน 0% 10เดือน Samsung ช่วยออก 2 เดื...,รายการส่งเสริมการขาย,OTHER,COM7,Telecom,,Samsung,False,Promotion Sale,Normal Product,Ordinary,Telecom Package,Telecom Package
5,0-083D71KJ,Network Card TB-21E,Network,,Unknown,Network,,Generic,False,Normal Sale,Normal Product,Ordinary,Network,Network
6,0000000000024,CASE TNC MAGNET MN02-KS 500W 24PIN 1FAN Airduc...,PC Case & Cooling,CASE<900,TNC,PC Case,,Generic,False,Normal Sale,Normal Product,Ordinary,IT Accessories,PC Case
7,000000000033012585,(Part) ND 720 CARE DISPLAY MODULE ASSY 00809K8...,SERVICE,SERVICE,SERVICE,SparePart,,Generic,False,Normal Sale,Normal Product,Ordinary,Spare Part,Spare Part
8,000000000052000488AIS,Film Autozkin for iPhone 6S Plus AIS,Phone Accessories,FILM IPHONE,Unknown,Film,Smart Phone,iPhone,False,Normal Sale,Normal Product,Ordinary,Smart Phone,Smart Phone Film
9,0000000000802,Case Plenty PL-EM80-KK 500W.1FAN 8CM.( Black ),PC Case & Cooling,CASE,PLENTY,PC Case,,Generic,False,Normal Sale,Normal Product,Ordinary,IT Accessories,PC Case


---

## สิ่งที่ยังค้าง

| | |
|---|---|
| **gold set** | ยังไม่มีใครตรวจ — ตัวเลขทุกตัวยังบอกได้แค่ "เหมือนกฎกี่ %" |
| **`Telecom / Package`** | ควรอยู่ใน `dim_item` ไหม ยังต้องถามเจ้าของระบบ ITEC |
| **`PROMO OPERATOR` · `BTB DEMO` · `RESERVE`** | ~15% ของทั้งชุดไม่ใช่สินค้าขายจริง ควรมี flag `is_sellable` แยก |
| **`PHYID`** | ยังไม่รู้ว่า map กับอะไร |

## ไฟล์ในโปรเจกต์นี้

```
scripts/itec/dimension/
├── itec_dimension.ipynb    ◄── ไฟล์หลัก · แก้กฎที่ Cell 3 ที่เดียว
├── dimension.py                เครื่องยนต์ · อ่าน rules.yaml
├── rules.yaml                  **generated — ห้ามแก้มือ**
├── _data/  dim_item_itec.csv
└── _out/   ผลลัพธ์ทั้งหมด
```

เกี่ยวข้อง → `ITEC Item Category Mapping (SQL to Python)` · `ITEC Category Toolkit` · `ITEC Model - Fine-tune Design`